# Cấp độ 2 — GRKT (KDD 2024) với tập con Junyi vừa phải và liên kết Cấp độ 1

Notebook này là bản **mới**, viết lại từ `kaggle_level2.ipynb` (bản đang chạy được, giữ nguyên
để đối chiếu). Ba thay đổi có chủ đích so với bản cũ:

1. **Junyi chỉ dùng tập con 3.000 người học**, không chạy toàn bộ 72.630 người / 16,2 triệu lượt.
   Mọi con số của Junyi trong notebook này là số của **tập con**, không phải "toàn bộ Junyi".
   Lý do: chi phí GPU. Số người chỉ ảnh hưởng thời gian (VRAM phụ thuộc batch × số khái niệm),
   nên lấy mẫu người học có seed cố định là cách giảm thời gian mà không đổi kiến trúc hay split
   theo người học.
2. **Liên kết Cấp độ 1 có kiểm chứng**: cùng seed split 42, cùng `max_seq_len=200`, cùng hàm
   `level1.pipeline.load_kt_data`; notebook đối chiếu tập người học với `outputs/level1/split.json`,
   so AUC đường nền với `outputs/level1/level1_metrics.csv`, và đo riêng các checkpoint Cấp độ 1
   đã commit để xử lý dứt điểm nghi vấn GAP_LEVEL2 §F2.
3. **Bổ sung sản phẩm còn thiếu**: provenance (`run_config.json`, `run_sessions.json`,
   `environment.txt`), dashboard người học Cấp độ 2, hiệu chỉnh ECE/Brier, và `report_summary.md`
   sinh tự động từ đĩa.

## 0.1 Đối chiếu yêu cầu và khoảng trống

| Yêu cầu / GAP | `kaggle_level2.ipynb` (bản cũ) | Notebook này |
|---|---|---|
| §5 tái lập GRKT trên ≥ 2 tập công khai | Có (assist2009 + junyi toàn bộ) | Có; Junyi là **tập con 3.000 người**, công bố rõ ở mọi bảng |
| §5 dùng lại đúng split theo người học của Cấp độ 1 | Có (seed 42) | Có, thêm kiểm tra tập người học trùng `level1/split.json` + đối chiếu AUC |
| §6.2 đồ thị chỉ dựng từ train | Có (`leak_report.txt`) | Giữ nguyên, trích số cạnh vào báo cáo |
| §6.4 tách theo người học | Có (assert rời nhau) | Giữ nguyên |
| §7 AUC / ACC / NLL | Có | Thêm `checkpoint_seed` cho mọi dòng và đánh giá checkpoint C1 cũ |
| §7 mean ± std ≥ 3 seed | Có (chỉ assist2009) | Giữ; Junyi 1 seed được ghi rõ, không giả vờ đa seed |
| §7 ablation đồ thị | Có (train vs zero trên assist2009) | Giữ; giải thích vì sao Junyi chỉ train |
| §7 một phân tích nâng cao | Có (quỹ đạo mastery) | Thêm ECE/Brier (bật `RUN_CALIBRATION=True`) |
| §8 dashboard người học Cấp độ 2 | **Thiếu** | §9: dashboard từ `traj_train.npz` + dự đoán từng bước, JSON/CSV/hình/dropdown |
| GAP F1 tiêu chí quỹ đạo bị đóng băng ở 1,0 | Đã sửa trong `level2/trajectory.py` | Notebook in rõ "kiểm chứng cài đặt" ≠ bằng chứng |
| GAP F2 checkpoint C1 bị ghi đè seed cuối | Có `checkpoint_seed` | Mặc định **train lại seed 42**; đo riêng checkpoint cũ để kết luận |
| GAP F3 artifact Junyi lệch tài liệu | Có ghi chú | Số liệu lần này là **tập con**, ghi ở mọi bảng và trong `report_summary.md` |
| GAP F4 GRKT 5 epoch vs baseline 20 | Có `BASELINE_EPOCHS` | GRKT và baseline cùng cap 20 epoch + cùng patience 5, ghi số epoch thực chạy |
| GAP F5 thiếu config/history/provenance | **Thiếu** | `run_config.json` (khoá cấu hình), `run_sessions.json` (history/resume), `environment.txt` |
| GAP F6 không có biến thể đồ thị | Có | Giữ (train/zero trên assist2009) |
| GAP F7 thiếu dashboard Cấp độ 2 | **Thiếu** | Đã thêm §9 |

### Phạm vi số liệu — đọc trước khi trích dẫn

- **assist2009**: toàn bộ split Cấp độ 1 (2.330 / 776 / 778 người học), `max_seq_len=200`.
- **junyi**: tập con **3.000 người học** lấy mẫu có seed 42 **sau** khi lọc ≥10 tương tác và giữ
  200 bài phổ biến nhất. Đây không phải toàn bộ Junyi. Việc chọn 200 khái niệm dùng thống kê trên
  toàn bộ log (transductive) — một giới hạn đã biết của `level2/dataset_spec.py`, được ghi lại
  trong báo cáo thay vì bỏ qua.
- **≥2 tập dữ liệu** vẫn đạt, nhưng kết luận về Junyi chỉ có giá trị trên tập con này.


## 0. Clone repo và chuẩn bị môi trường

Giống Cấp độ 1: clone bản `main`, đưa repo vào `sys.path`, xoá module cũ khỏi `sys.modules`
để chạy lại notebook từ giữa vẫn ăn code mới, và **chỉ cài các gói còn thiếu** (không
`pip install torch` — bản CPU sẽ phá môi trường GPU của Kaggle).


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/linhnnh688/Knowledge-Tracing.git"
# Nhánh GitHub để clone. Đổi sang nhánh khác (vd "main" sau khi merge) khi chạy thử;
# code phải được PUSH lên GitHub trước, Kaggle không thấy thay đổi chỉ có ở máy local.
REPO_BRANCH = "features/clean-dry-enhancement"
KAGGLE_ROOT = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_ROOT.exists()

if IS_KAGGLE:
    REPO = KAGGLE_ROOT / "Knowledge-Tracing"
    if not (REPO / ".git").exists():
        if REPO.exists():
            shutil.rmtree(REPO, ignore_errors=True)
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO)])
    else:
        try:
            subprocess.check_call(
                ["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH, "--depth", "1"]
            )
            subprocess.check_call(
                ["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"]
            )
        except Exception as exc:
            print(f"Git sync lỗi ({exc}) -> clone lại từ đầu")
            shutil.rmtree(REPO, ignore_errors=True)
            subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO)])
else:
    REPO = (
        Path.cwd()
        if (Path.cwd() / "level2").exists()
        else Path.cwd() / "Knowledge-Tracing"
    )

os.chdir(REPO)
# GKT/ là package (import tương đối) nên chỉ cần REPO trên sys.path.
for p in [REPO]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Code trên đĩa vừa được git cập nhật; module đã import trước đó vẫn nằm trong sys.modules.
_stale = [
    m
    for m in list(sys.modules)
    if m.split(".")[0] in ("level1", "level2", "GRKT", "GKT", "kt_common")
]
for m in _stale:
    sys.modules.pop(m, None)
print(f"Đã bỏ {len(_stale)} module dự án cũ khỏi sys.modules.")

# outputs/level1/split.json là Git LFS: clone --depth 1 có thể chỉ nhận file con trỏ.
# split.json cần cho phép đối chiếu tập người học, nên thử lấy bản thật (best-effort).
_level1_dir = REPO / "outputs" / "level1"
_lfs_pointer = []
for _p in (_level1_dir / "split.json", _level1_dir / "metrics.json"):
    if _p.exists():
        try:
            if (
                _p.read_text(encoding="utf-8", errors="replace")
                .lstrip()
                .startswith("version https://git-lfs")
            ):
                _lfs_pointer.append(_p.name)
        except OSError:
            pass
if _lfs_pointer:
    print("Phát hiện file LFS chưa fetch:", _lfs_pointer)
    try:
        subprocess.check_call(
            ["git", "-C", str(REPO), "lfs", "pull", "--include", "outputs/level1/*"],
            stdout=subprocess.DEVNULL,
        )
    except Exception as exc:
        print(
            f"[cảnh báo] không fetch được LFS ({exc}); đối chiếu split.json sẽ được bỏ qua có báo cáo."
        )
    else:
        print("Đã chạy git lfs pull cho outputs/level1/*.")

try:
    _head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "--oneline", "-1"], text=True
    ).strip()
except Exception as exc:
    _head = f"(không đọc được git: {exc})"
print("HEAD đang dùng:", _head)
print("Repo:", REPO.resolve())
print(
    "level2 package:",
    (REPO / "level2").is_dir(),
    "| GRKT vendored:",
    (REPO / "GRKT").is_dir(),
)
print("outputs/level1:", _level1_dir.is_dir())

In [ ]:
import importlib.util
import subprocess
import sys

OPTIONAL = {
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}
missing = [
    pkg for mod, pkg in OPTIONAL.items() if importlib.util.find_spec(mod) is None
]
if missing:
    print("Cài bổ sung (chỉ gói chưa có):", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Mọi thư viện tuỳ chọn đã có sẵn — không cài gì thêm.")

import matplotlib as _mpl
import numpy as _np
import pandas as _pd

print(
    "python",
    sys.version.split()[0],
    "| numpy",
    _np.__version__,
    "| pandas",
    _pd.__version__,
    "| matplotlib",
    _mpl.__version__,
)

## 0.2 Cấu hình thực nghiệm

Các giá trị dưới đây được chọn theo ba ràng buộc:

- **trung thực với yêu cầu**: cùng seed split 42 và `max_seq_len=200` với Cấp độ 1, GRKT và
  baseline cùng cap 20 epoch + cùng patience 5;
- **vừa phải về thời gian**: Junyi chỉ 3.000 người; mỗi biến thể đồ thị là một lần train riêng
  nên ablation chỉ chạy trên assist2009;
- **không trộn cấu hình**: `run_config.json` khoá các tham số ảnh hưởng kết quả. Resume với cấu
  hình khác sẽ raise thay vì lặng lẽ ghép hai lần chạy (GAP F5).

- **đánh giá lại không train**: `EVAL_ONLY=True` + attach `level2_moderate_output.zip` -> §3 kiểm
  tra đủ checkpoint GRKT, đa seed và đường nền rồi nạp để đánh giá; thiếu file nào in đúng đường
  dẫn thay vì âm thầm train lại.


In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from level2.dataset_spec import (
    ensure_canonical,
    find_kaggle_input,
    get_spec,
    probe_columns,
)
from level2.experiment import BASELINE_SPECS, run_level2
from kt_common.checkpoint import checkpoint_path, find_checkpoint

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NOTEBOOK_VERSION = "2026-11-12 (order-dedup + bn-eval; đóng gói định kỳ)"
QUICK_RUN = (
    False  # True = chỉ assist2009_smoke, 2 epoch: kiểm tra mã, không lấy số liệu
)

DATASETS = ["assist2009_smoke"] if QUICK_RUN else ["assist2009", "junyi"]

# ---------------------------------------------------------------- Junyi: tập con vừa phải
# 72.630 người / 16,2 triệu lượt là quá đắt cho một phiên Kaggle (xem GAP_LEVEL2 §F3).
# Lấy mẫu 3.000 người có seed (spec.subsample_seed=42) sau khi lọc >=10 tương tác và giữ
# top-200 bài phổ biến. Số người chỉ ảnh hưởng THỜI GIAN, không ảnh hưởng VRAM.
JUNYI_MAX_USERS = 300 if QUICK_RUN else 3000
JUNYI_TOP_CONCEPTS = 200
JUNYI_MIN_INTERACTIONS = 10
MAX_USERS = JUNYI_MAX_USERS

EPOCHS = 2 if QUICK_RUN else 20  # GRKT: cùng cap với baseline (đóng GAP F4)
PATIENCE = 2 if QUICK_RUN else 5  # cùng patience với train_baseline_resumable
BASELINE_EPOCHS = 2 if QUICK_RUN else 20  # đường nền DKT/GKT: đúng công thức Cấp độ 1
BATCH_SIZE = 16 if QUICK_RUN else 32
MAX_SEQ_LEN = 50 if QUICK_RUN else 200  # 200 = đúng cấu hình Cấp độ 1
SPLIT_SEED = 42  # BẤT DI BẤT DỊCH: seed split của Cấp độ 1
LR = 1e-3
D_HIDDEN, K_HIDDEN, K_HOP = 128, 16, 1
GRAPH_THRESH = 0.6
COLD_THRESH = 1 if QUICK_RUN else 5
ALPHA = 0.01
GAP_CAP_MIN = 240

RUN_BASELINES = True
# Mỗi biến thể là một lần train GRKT: chỉ bật zero trên tập rẻ (assist2009). P0-2
# (train vs zero) đã đủ cho yêu cầu §7; Junyi giữ một đồ thị train và điều đó được ghi rõ.
GRAPH_ABLATION_DATASETS = ("assist2009",) if not QUICK_RUN else ()
VARIANTS_BY_DATASET = {
    ds: (("train", "zero") if ds in GRAPH_ABLATION_DATASETS else ("train",))
    for ds in DATASETS
}
RUN_MULTI_SEED = not QUICK_RUN
MULTI_SEED_DATASETS = tuple(ds for ds in ("assist2009",) if ds in DATASETS)
SEEDS = [42, 123, 456]
RECORD_TRAJECTORY = True
RUN_CALIBRATION = (
    True  # ECE/Brier chỉ là inference nên rẻ; đủ cho "một phân tích nâng cao"
)
DASHBOARD_MAX_LEARNERS = 5 if QUICK_RUN else 20

# P0/F2: outputs/level1/*.pt được commit ngày 16/10, TRƯỚC bản sửa tên checkpoint theo seed
# (01/11) -> không chứng minh được chúng là seed 42. Mặc định train lại đường nền với seed 42;
# checkpoint cũ vẫn được ĐO riêng ở §5.2 (chỉ để chẩn đoán, không dùng cho bảng chính).
USE_COMMITTED_CKPTS = False
EVAL_COMMITTED_CKPTS = True

RESUME_GRKT_CKPT = True
# True = CHỈ đánh giá + dashboard từ output đã đính kèm; không train thêm run nào.
# Cần level2_moderate_output.zip (có <tập>/GRKT-*.pt, checkpoint đa seed và baselines/).
EVAL_ONLY = False
if EVAL_ONLY and not RESUME_GRKT_CKPT:
    print("[EVAL_ONLY] bật RESUME_GRKT_CKPT=True để nạp checkpoint thay vì train lại.")
    RESUME_GRKT_CKPT = True
ALLOW_CONFIG_MISMATCH = (
    False  # True chỉ khi chủ động muốn trộn cấu hình (không khuyến nghị)
)
CKPT_EVERY_MIN = 10
# Đóng gói zip ở LUỒNG NỀN, cách nhau chừng này phút (0 = tắt). Kaggle có thể kill phiên
# cứng (hết quota GPU 30h/tuần, rớt kết nối, quá 12h) mà KHÔNG chạy khối except của §3, nên
# chỉ đóng gói lúc dừng vẫn còn cửa mất hàng giờ checkpoint. 20 phút = tổn thất tối đa khi
# mất phiên, và đủ thưa để việc nén ~150 MB không giành CPU của dataloader.
PACK_EVERY_MIN = 5 if QUICK_RUN else 20
SESSION_LIMIT_H = 12
SESSION_SAFETY_MIN = 45
# Ngân sách của CẢ cell §3 (phút). Lần đầu nên đặt cao; phiên resume có thể hạ còn 240-300.
TIME_BUDGET_MIN = 30 if QUICK_RUN else 540
RESUME_ZIP = None  # None = tự dò /kaggle/input; hoặc "/kaggle/input/<ds>/level2_moderate_output.zip"

TOP = "level2_moderate"
SAVE_DIR = Path("/kaggle/working") / TOP if IS_KAGGLE else (REPO / "outputs" / TOP)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LEVEL1_DIR = REPO / "outputs" / "level1"
ZIP_PATH = (
    Path("/kaggle/working") if IS_KAGGLE else REPO
) / f"{TOP}_output.zip"
REPORT_ZIP_PATH = (
    Path("/kaggle/working") if IS_KAGGLE else REPO
) / f"{TOP}_report_pack.zip"
FIG_DIR = SAVE_DIR / "figures"
TAB_DIR = SAVE_DIR / "tables"


def _session_start_epoch():
    """Mốc bắt đầu phiên Kaggle = lúc PID 1 khởi động; None nếu không đọc được."""
    try:
        fields = Path("/proc/1/stat").read_text().rsplit(")", 1)[1].split()
        start_ticks = float(fields[19])
        uptime = float(Path("/proc/uptime").read_text().split()[0])
        return time.time() - uptime + start_ticks / os.sysconf("SC_CLK_TCK")
    except Exception:
        return None


_session_start = _session_start_epoch() if IS_KAGGLE else None
if TIME_BUDGET_MIN and _session_start is not None:
    _elapsed_min = (time.time() - _session_start) / 60.0
    _left_min = SESSION_LIMIT_H * 60 - _elapsed_min - SESSION_SAFETY_MIN
    if not (0.0 <= _elapsed_min <= SESSION_LIMIT_H * 60):
        print(
            f"[cảnh báo] đồng hồ phiên không hợp lệ ({_elapsed_min:.0f} phút) -> bỏ chặn theo đồng hồ."
        )
    else:
        print(
            f"Phiên Kaggle: đã chạy {_elapsed_min:.0f} phút | còn {_left_min:.0f} phút tới chốt "
            f"{SESSION_LIMIT_H}h (đã trừ {SESSION_SAFETY_MIN} phút an toàn)"
        )
        TIME_BUDGET_MIN = max(1.0, min(float(TIME_BUDGET_MIN), _left_min))

# Các tham số ảnh hưởng trực tiếp tới số liệu. Resume với bộ này khác đi sẽ bị chặn ở §0.3.
from kt_common.interactions import DATA_VERSION

RUN_CONFIG_LOCKED = {
    # Zip/checkpoint của tiền xử lý cũ (dòng trùng, sai thứ tự) bị chặn khi resume.
    "data_version": DATA_VERSION,
    "datasets": list(DATASETS),
    "split_seed": int(SPLIT_SEED),
    "max_seq_len": int(MAX_SEQ_LEN),
    "epochs": int(EPOCHS),
    "patience": int(PATIENCE),
    "baseline_epochs": int(BASELINE_EPOCHS),
    "batch_size": int(BATCH_SIZE),
    "lr": float(LR),
    "d_hidden": int(D_HIDDEN),
    "k_hidden": int(K_HIDDEN),
    "k_hop": int(K_HOP),
    "alpha": float(ALPHA),
    "graph_thresh": float(GRAPH_THRESH),
    "cold_thresh": int(COLD_THRESH),
    "gap_cap_min": int(GAP_CAP_MIN),
    "max_users": None if MAX_USERS is None else int(MAX_USERS),
    "seeds": [int(s) for s in SEEDS],
    "variants_by_dataset": {k: list(v) for k, v in VARIANTS_BY_DATASET.items()},
    "multi_seed_datasets": list(MULTI_SEED_DATASETS),
    "run_baselines": bool(RUN_BASELINES),
    "run_calibration": bool(RUN_CALIBRATION),
    "use_committed_ckpts": bool(USE_COMMITTED_CKPTS),
}

print(f"Notebook {NOTEBOOK_VERSION}")
print(f"  tập dữ liệu       : {DATASETS}")
print(
    f"  GRKT              : epochs={EPOCHS} patience={PATIENCE} batch={BATCH_SIZE} "
    f"max_seq_len={MAX_SEQ_LEN} K_hidden={K_HIDDEN}"
)
print(f"  baseline          : epochs={BASELINE_EPOCHS} (train lại seed {SPLIT_SEED})")
print(f"  biến thể đồ thị   : {VARIANTS_BY_DATASET}")
print(f"  đa seed           : {MULTI_SEED_DATASETS or 'không'} | seeds={SEEDS}")
print(
    f"  Junyi             : cap {MAX_USERS} người | top_concepts={JUNYI_TOP_CONCEPTS} "
    f"| min_interactions={JUNYI_MIN_INTERACTIONS} (KHÔNG phải toàn bộ)"
)
print(
    f"  ngân sách cell §3 : {TIME_BUDGET_MIN:.0f} phút | calibration={RUN_CALIBRATION}"
)
print(
    f"  đóng gói định kỳ  : mỗi {PACK_EVERY_MIN} phút -> {ZIP_PATH.name}"
    + (" (TẮT)" if not PACK_EVERY_MIN else " (ghi nguyên tử .part)")
)
print(
    f"  chế độ            : {'EVAL_ONLY (chỉ đánh giá + dashboard)' if EVAL_ONLY else 'train/resume'}"
)
print(f"  output            : {SAVE_DIR}")
print(f"  Level-1 reference : {LEVEL1_DIR} | tồn tại: {LEVEL1_DIR.is_dir()}")

# ---------------------------------------------------------------- chặn cấu hình sai sớm
_problems = []
if not QUICK_RUN:
    _assist_id = "assist2009"
    if _assist_id in DATASETS:
        if "zero" not in VARIANTS_BY_DATASET.get(_assist_id, ()):
            _problems.append(
                "assist2009 thiếu biến thể 'zero' (P0-2 / §7 ablation đồ thị)"
            )
        if _assist_id not in MULTI_SEED_DATASETS:
            _problems.append(
                "assist2009 không nằm trong MULTI_SEED_DATASETS (P0-3 / §7 đa seed)"
            )
    if MAX_SEQ_LEN != 200:
        _problems.append(
            f"MAX_SEQ_LEN={MAX_SEQ_LEN} khác Cấp độ 1 (200) -> không đối chiếu được"
        )
    if int(SPLIT_SEED) != 42:
        _problems.append(f"SPLIT_SEED={SPLIT_SEED} khác Cấp độ 1 (42)")
    if USE_COMMITTED_CKPTS:
        _problems.append(
            "USE_COMMITTED_CKPTS=True -> seed checkpoint C1 không xác định (P0-4)"
        )
    if MAX_USERS is None or int(MAX_USERS) > 5000:
        _problems.append(f"MAX_USERS={MAX_USERS} không phải 'vừa phải' (đặt <= 5000)")
    if get_spec("junyi").top_concepts != JUNYI_TOP_CONCEPTS:
        _problems.append("JUNYI_TOP_CONCEPTS không khớp spec -> tập con sẽ khác mô tả")
    if get_spec("junyi").min_interactions != JUNYI_MIN_INTERACTIONS:
        _problems.append(
            "JUNYI_MIN_INTERACTIONS không khớp spec -> tập con sẽ khác mô tả"
        )
    if get_spec("assist2009").min_interactions != 3:
        _problems.append("assist2009 min_interactions != 3 -> split sẽ khác Cấp độ 1")
print("[P0 CHECK]", "OK" if not _problems else " | ".join(_problems))
if _problems:
    raise RuntimeError("Cấu hình chưa hợp lệ: " + "; ".join(_problems))

## 0.3 Khôi phục phiên trước và khoá cấu hình

Ba việc, theo đúng thứ tự:

1. **Khôi phục** zip/cây kết quả của phiên trước (Kaggle tự giải nén zip trong dataset, nên cả
   hai dạng đều được nhận). Kiểm tra zip-slip trước khi ghi.
2. **Cách ly artifact Junyi của lần chạy toàn bộ** nếu lỡ attach nhầm: chúng được chuyển vào
   `junyi/stale_<thời-điểm>/` chứ không xoá, để vẫn đối chiếu được (F3).
3. **Khoá cấu hình**: nếu `run_config.json` của lần trước khác cấu hình hiện tại thì raise. Không
   có bước này, resume sau khi đổi `EPOCHS`/`max_users` sẽ trộn hai lần chạy và bảng vẫn trông
   hợp lệ (GAP F5).


In [ ]:
# ==== 0.3 Khôi phục + cách ly + khoá cấu hình ====
import json
import time
import zipfile


def _is_newer_or_equal(target: Path, src_mtime: float) -> bool:
    return target.exists() and target.stat().st_mtime >= src_mtime


def _restore_from_zip(path: Path) -> int:
    n = 0
    with zipfile.ZipFile(path) as zf:
        infos = [i for i in zf.infolist() if not i.is_dir()]
        unsafe = [
            i.filename
            for i in infos
            if i.filename.startswith("/") or ".." in Path(i.filename).parts
        ]
        if unsafe:
            raise RuntimeError(f"zip chứa đường dẫn không an toàn: {unsafe[:5]}")
        prefix = TOP + "/"
        if not any(i.filename.startswith(prefix) for i in infos):
            raise RuntimeError(
                f"{path} không phải zip của notebook này (thiếu tiền tố {prefix})"
            )
        for info in infos:
            target = SAVE_DIR / Path(info.filename).relative_to(TOP)
            if _is_newer_or_equal(target, time.mktime(info.date_time + (0, 0, -1))):
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
            n += 1
    return n


def _restore_from_tree(src: Path) -> int:
    n = 0
    for p in src.rglob("*"):
        if not p.is_file():
            continue
        target = SAVE_DIR / p.relative_to(src)
        if _is_newer_or_equal(target, p.stat().st_mtime):
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, target)
        n += 1
    return n


_resume_zip, _resume_tree = None, None
if RESUME_ZIP:
    _p = Path(RESUME_ZIP)
    if not _p.exists():
        raise FileNotFoundError(
            f"RESUME_ZIP={RESUME_ZIP} không tồn tại — kiểm tra Add Input"
        )
    _resume_zip, _resume_tree = (_p, None) if _p.is_file() else (None, _p)
else:
    _root = Path("/kaggle/input")
    _zips = sorted(_root.glob("*/level2_moderate_output.zip")) if _root.is_dir() else []
    _trees = sorted(p for p in _root.rglob(TOP) if p.is_dir()) if _root.is_dir() else []
    if len(_zips) == 1:
        _resume_zip = _zips[0]
    elif len(_zips) > 1:
        print("Có nhiều zip — đặt RESUME_ZIP cho rõ ràng:")
        for c in _zips:
            print("   ", c)
    elif len(_trees) == 1:
        _resume_tree = _trees[0]
    elif len(_trees) > 1:
        print(f"Có nhiều thư mục {TOP} — đặt RESUME_ZIP cho rõ ràng:")
        for c in _trees:
            print("   ", c)

if _resume_zip is not None:
    print(f"Đã phục hồi {_restore_from_zip(_resume_zip)} file từ {_resume_zip}")
elif _resume_tree is not None:
    print(f"Đã phục hồi {_restore_from_tree(_resume_tree)} file từ cây {_resume_tree}")
else:
    print(
        "Không có zip/cây resume trong /kaggle/input -> lần này train từ đầu (nếu chưa có ckpt)."
    )


def _quarantine_junyi() -> None:
    """Chuyển artifact Junyi của lần chạy toàn bộ sang thư mục stale_* (KHÔNG xoá)."""
    d = SAVE_DIR / "junyi"
    if not d.is_dir() or MAX_USERS is None:
        return
    reasons = []
    n_users = None
    split_path = d / "split.json"
    if split_path.exists():
        try:
            obj = json.loads(split_path.read_text(encoding="utf-8"))
            n_users = sum(
                len(obj[k])
                for k in ("train_users", "val_users", "test_users")
                if k in obj
            )
        except Exception:
            n_users = None
    if n_users is not None and n_users > MAX_USERS * 1.2:
        reasons.append(f"split có {n_users} người > cap {MAX_USERS}")
    ckpt = find_checkpoint(d, "GRKT-train", SPLIT_SEED) or checkpoint_path(
        d, "GRKT-train", SPLIT_SEED
    )
    if ckpt.exists():
        try:
            payload = torch.load(ckpt, map_location="cpu", weights_only=False)
            fp = payload.get("fingerprint") if isinstance(payload, dict) else None
            n_train = fp.get("n_train") if isinstance(fp, dict) else None
            if n_train and n_train > MAX_USERS * 0.6 * 1.3:
                reasons.append(f"checkpoint train trên {n_train} người")
        except Exception:
            pass
    meta = d / "junyi_canonical.csv.meta.json"
    if meta.exists():
        try:
            m = json.loads(meta.read_text(encoding="utf-8"))
            if m.get("max_users") != MAX_USERS:
                reasons.append(
                    f"CSV chuẩn hoá max_users={m.get('max_users')} != {MAX_USERS}"
                )
        except Exception:
            pass
    if not reasons:
        return
    dest = d / f"stale_{time.strftime('%Y%m%d_%H%M%S')}"
    dest.mkdir(parents=True, exist_ok=True)
    moved = []
    for p in sorted(d.iterdir()):
        if p.resolve() == dest.resolve() or p.name.startswith("stale_"):
            continue
        shutil.move(str(p), str(dest / p.name))
        moved.append(p.name)
    (dest / "QUARANTINE.json").write_text(
        json.dumps(
            {"reasons": reasons, "moved": moved, "notebook": NOTEBOOK_VERSION},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"[cách ly Junyi] {reasons} -> đã chuyển {len(moved)} mục vào {dest}")


_quarantine_junyi()


def _current_locked() -> dict:
    return json.loads(json.dumps(RUN_CONFIG_LOCKED, sort_keys=True, default=str))


_lock_path = SAVE_DIR / "run_config.json"
_prev_locked = None
if _lock_path.exists():
    try:
        _prev_locked = json.loads(_lock_path.read_text(encoding="utf-8")).get("locked")
    except Exception:
        _prev_locked = None

if _prev_locked is not None and _prev_locked != _current_locked():
    _diff = sorted(
        k for k in _current_locked() if _prev_locked.get(k) != _current_locked()[k]
    )
    _msg = f"run_config.json khác cấu hình hiện tại ở: {_diff}"
    if not ALLOW_CONFIG_MISMATCH:
        raise RuntimeError(
            _msg + ". Đổi cấu hình mà vẫn dùng thư mục cũ sẽ trộn hai lần chạy. "
            "Chọn một: (a) đổi SAVE_DIR/TOP cho lần chạy mới, (b) đặt ALLOW_CONFIG_MISMATCH=True "
            "nếu thật sự muốn ghép, hoặc (c) xoá thư mục output cũ."
        )
    print("[cảnh báo]", _msg)
elif _prev_locked is None:
    _has_ckpt = any(SAVE_DIR.rglob("GRKT-*.pt")) or any(SAVE_DIR.rglob("ckpt_*.pt"))
    if _has_ckpt and not ALLOW_CONFIG_MISMATCH:
        raise RuntimeError(
            f"Có checkpoint trong {SAVE_DIR} nhưng không có run_config.json -> không chứng minh được "
            "chúng cùng cấu hình. Đặt ALLOW_CONFIG_MISMATCH=True nếu chấp nhận, hoặc dùng thư mục mới."
        )

_lock_path.write_text(
    json.dumps(
        {
            "notebook": NOTEBOOK_VERSION,
            "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "locked": _current_locked(),
            "dataset_caps": {
                "junyi_max_users": MAX_USERS,
                "junyi_top_concepts": JUNYI_TOP_CONCEPTS,
                "junyi_min_interactions": JUNYI_MIN_INTERACTIONS,
            },
            "time_budget_min": float(TIME_BUDGET_MIN or 0),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# Ghi một mục "đang chạy" vào nhật ký phiên; §4 sẽ điền số liệu vào chính mục cuối này.
_sessions_path = SAVE_DIR / "run_sessions.json"
_sessions = []
if _sessions_path.exists():
    try:
        _sessions = json.loads(_sessions_path.read_text(encoding="utf-8"))
    except Exception:
        _sessions = []
_sessions.append(
    {
        "notebook": NOTEBOOK_VERSION,
        "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "git_commit": _head,
        "device": str(device),
        "status": "started",
        "locked": _current_locked(),
    }
)
_sessions_path.write_text(
    json.dumps(_sessions, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Đã khoá cấu hình -> {_lock_path}")
print(f"Nhật ký phiên   -> {_sessions_path} ({len(_sessions)} phiên)")

## 1. GRKT làm gì, và bản vendored đã sửa gì

GRKT (Graph-based Retrieval-Knowledge Tracing, KDD 2024) chia một bước làm bài thành ba giai
đoạn trên đồ thị khái niệm:

- **truy xuất (retrieval)**: lan truyền trạng thái ẩn của các khái niệm lân cận qua `rel_map` /
  `pre_map` để tạo `m_tilde` — vector thật sự đi vào đầu dự đoán;
- **củng cố (strengthening)**: đúng thì cộng "gain", sai thì trừ "loss", tách biệt hai loại cạnh;
- **học / quên (learning-forgetting)**: dịch trạng thái theo khoảng thời gian giữa hai bước.

Vì ba giai đoạn tách bạch, quỹ đạo `m(u, c, t)` đo được từng phần, và đó là "một phân tích nâng
cao" mà `YEU_CAU.md` §7 yêu cầu.

### Bảy sửa đổi so với upstream (`GRKT/`)

| Sửa đổi | Lý do |
|---|---|
| Import tương đối (`from .model...`) | `GRKT/` nằm cạnh `level1/`, `level2/` trong cùng repo |
| Bỏ `Runner.py`, train theo split người học của Cấp độ 1 | Runner chia k-fold trên cửa sổ 100 bước nên cùng người học xuất hiện ở train và test (vi phạm §6.4) |
| `rel_map`/`pre_map` chỉ đếm từ train | Chống rò rỉ (§6.2) |
| Trục thời gian cộng dồn theo phút, chặn 240 phút | Junyi có timestamp tuyệt đối; ASSIST chỉ có thời lượng làm bài |
| Ngưỡng nhị phân 0,5, `k_hop=1`, `Positive_Linear` | Giữ nguyên kiến trúc bài báo; các lựa chọn này khiến ba tiêu chí `consistency_*`/`locality` bị ép ≡ 1 |
| Hook `rec` ghi `m_pre`/`m_tilde`/`m_strong` | Phục vụ nhiệm vụ 2 (quỹ đạo) mà không đổi forward |
| Registry mô hình + allowlist | Chỉ dựng đúng lớp GRKT, không chạy pickle tuỳ ý từ checkpoint |

Hệ quả quan trọng khi viết báo cáo: **`consistency_strengthen`, `consistency_spread`, `locality`
là kiểm chứng cài đặt, không phải bằng chứng vai trò đồ thị** (GAP F1). Bằng chứng thật là
`precedence_gap`, `stability_spearman`, `forget_slope`, `pearson_dt_dm`, cùng so sánh
`train` vs `zero`.


## 2. EDA và kiểm tra dữ liệu trước khi chuẩn hoá

Junyi bắt buộc phải Agree/Subscribe trên Kaggle rồi Add Input; notebook tự dò file trong
`/kaggle/input`. Cell này in **header thật** của file gốc (tên cột đổi thì thấy ngay), rồi chuẩn
hoá với đúng cap của tập con và in quy mô thực tế.


In [ ]:
from level2.dataset_spec import list_kaggle_inputs

# Số liệu công bố của Junyi (Log_Problem.csv) — chỉ để so quy mô, KHÔNG phải dữ liệu lần này.
JUNYI_PUBLISHED = {
    "rows": 16_217_311,
    "users": 72_630,
    "exercises": 1_326,
    "source": "Kaggle junyiacademy/learning-activity-public-dataset-by-junyi-academy (Log_Problem.csv)",
}
print("Junyi công bố:", JUNYI_PUBLISHED)

for name in DATASETS:
    spec = get_spec(name)
    print("=" * 72)
    print(f"[{name}] {spec.note}")
    if name == "junyi":
        guess = find_kaggle_input(patterns=("junyi", "log_problem"))
        print("  file gốc dò được:", guess)
        if guess is None:
            print("  Hiện trạng /kaggle/input:")
            for line in list_kaggle_inputs().splitlines():
                print("    " + line)
            print(
                "  >> Cần Agree/Subscribe dataset Junyi rồi Add Input; cell §3 sẽ dừng nếu thiếu."
            )
            continue
        with open(guess, encoding="utf-8-sig", errors="replace") as fh:
            print(f"  cột gốc của {guess.name}: {fh.readline().strip()}")
    try:
        canonical = ensure_canonical(
            spec, out_dir=SAVE_DIR / name, max_users=MAX_USERS, verbose=True
        )
    except FileNotFoundError as exc:
        print(f"  [bỏ qua {name}] thiếu dữ liệu:\n{exc}")
        continue
    print("  CSV chuẩn hoá:", canonical)
    print("  cột chuẩn hoá:", probe_columns(canonical))

    eda = pd.read_csv(canonical)
    n_users = int(eda["user_id"].nunique())
    print(
        f"  {len(eda):,} tương tác | {n_users:,} người học | "
        f"{eda['skill_id'].nunique():,} khái niệm | {eda['problem_id'].nunique():,} câu hỏi | "
        f"tỉ lệ đúng {eda['correct'].mean():.3f}"
    )
    if name == "junyi":
        print(
            f"  [TẬP CON] {n_users:,} / {JUNYI_PUBLISHED['users']:,} người "
            f"({n_users / JUNYI_PUBLISHED['users']:.1%}), "
            f"{len(eda):,} / {JUNYI_PUBLISHED['rows']:,} lượt "
            f"({len(eda) / JUNYI_PUBLISHED['rows']:.1%})."
        )
        print(
            f"  [TẬP CON] cap={MAX_USERS} (subsample_seed={spec.subsample_seed}), "
            f"top_concepts={JUNYI_TOP_CONCEPTS}, min_interactions={JUNYI_MIN_INTERACTIONS}."
        )
        print(
            "  [GIỚI HẠN] top_concepts chọn trên TOÀN BỘ log (transductive); "
            "mọi số liệu Junyi dưới đây là của TẬP CON, không phải toàn bộ Junyi."
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    axes[0].hist(
        eda.groupby("user_id").size().to_numpy(), bins=40, edgecolor="black", alpha=0.7
    )
    axes[0].set_title(f"{name}: số tương tác mỗi người học")
    axes[0].set_xlabel("số lượt")
    top_skill = eda["skill_id"].value_counts().head(20).iloc[::-1]
    axes[1].barh(
        [str(x) for x in top_skill.index], top_skill.to_numpy(), color="#4878a6"
    )
    axes[1].set_title(f"{name}: 20 khái niệm thường gặp nhất")
    axes[1].set_xlabel("số lượt")
    fig.tight_layout()
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"eda_{name}.png", dpi=140, bbox_inches="tight")
    plt.show()

## 3. Chuỗi thực nghiệm Cấp độ 2

`run_level2` chạy trọn pipeline cho từng tập: chuẩn hoá → split theo người học → chuỗi GRKT →
đồ thị train-only → GRKT → đường nền Cấp độ 1 → quỹ đạo → hiệu chỉnh → đa seed. Cell bọc ba
tình huống:

- **hết `TIME_BUDGET_MIN`**: `TrainingPaused` đã ghi checkpoint trước khi raise; cell tự đóng gói
  zip rồi raise tiếp. Phiên sau attach zip, giữ `RESUME_GRKT_CKPT=True`, chạy lại.
- **lỗi bất ngờ** (OOM, mất mạng): vẫn đóng gói những gì đã có.
- **chạy xong**: ghi bảng nối của mọi tập có trong `SAVE_DIR` (không chỉ tập của lần này)
  rồi tự đóng `level2_moderate_output.zip`. Zip cũng được đóng ngay sau **mỗi tập**, nên
  kết quả assist2009 không mất nếu junyi hết ngân sách hoặc phiên bị kill.

Bao trùm cả ba tình huống là `PeriodicPackager`: một luồng nền đóng gói lại zip mỗi
`PACK_EVERY_MIN` phút trong suốt thời gian `run_level2` chạy. Nó tồn tại vì Kaggle có thể kill
phiên **cứng** (hết quota GPU, rớt kết nối, quá 12 h) mà không cho khối `except` chạy, khi đó
hai nhánh trên không kịp đóng gói. Zip ghi nguyên tử (`<tên>.part` rồi `os.replace`) nên một
lần đóng gói bị cắt giữa chừng chỉ mất file `.part`, không phá zip tốt đang có.


In [ ]:
from level2.archive import PeriodicPackager, package_output
from level2.grkt_api import TrainingPaused

if EVAL_ONLY:
    _need = []
    for _ds in DATASETS:
        _vars = tuple(VARIANTS_BY_DATASET.get(_ds, ("train",)))
        for _v in _vars:
            _p = find_checkpoint(
                SAVE_DIR / _ds, f"GRKT-{_v}", SPLIT_SEED
            ) or checkpoint_path(SAVE_DIR / _ds, f"GRKT-{_v}", SPLIT_SEED)
            if not _p.exists():
                _need.append(_p.relative_to(SAVE_DIR).as_posix())
        if _ds in MULTI_SEED_DATASETS:
            _kind = "train" if "train" in _vars else _vars[0]
            for _s in SEEDS:
                if int(_s) == int(SPLIT_SEED):
                    continue
                _p = SAVE_DIR / _ds / f"GRKT-{_kind}-seed{int(_s)}.pt"
                if not _p.exists():
                    _need.append(_p.relative_to(SAVE_DIR).as_posix())
    if RUN_BASELINES and not USE_COMMITTED_CKPTS:
        _base_names = [mt if mt == "DKT" else f"GKT-{gt}" for mt, gt in BASELINE_SPECS]
        for _ds in DATASETS:
            for _n in _base_names:
                _p = find_checkpoint(
                    SAVE_DIR / _ds / "baselines", _n, SPLIT_SEED
                ) or checkpoint_path(SAVE_DIR / _ds / "baselines", _n, SPLIT_SEED)
                if not _p.exists():
                    _need.append(_p.relative_to(SAVE_DIR).as_posix())
    if _need:
        raise RuntimeError(
            "EVAL_ONLY=True nhưng thiếu " + ", ".join(_need) + f". Attach {ZIP_PATH.name} "
            "vào /kaggle/input rồi chạy lại; hoặc đặt EVAL_ONLY=False để huấn luyện tiếp."
        )
    print(
        f"[EVAL_ONLY] đủ checkpoint GRKT/đa seed/đường nền trong {SAVE_DIR} -> "
        "chỉ đánh giá + dashboard, không train."
    )


# Luồng nền đóng gói mỗi PACK_EVERY_MIN phút: phiên Kaggle có thể bị kill cứng mà KHÔNG chạy
# khối except bên dưới, nên chỉ đóng gói lúc dừng thì vẫn còn cửa mất hàng giờ checkpoint.
# Zip ghi nguyên tử (.part -> os.replace) nên một lần đóng gói dang dở không phá zip tốt của
# lần trước; khoá theo đường dẫn zip nên luồng nền và _package_now không ghi đè nhau.
packager = PeriodicPackager(
    SAVE_DIR,
    ZIP_PATH,
    every_min=PACK_EVERY_MIN,
    pkg_kwargs={"max_mb": 150, "keep_ckpts": True},
)


def _package_now(reason: str) -> None:
    """Đóng gói ngay, in kiểm kê zip — Kaggle xoá /kaggle/working khi phiên kết thúc.

    Gọi trực tiếp ``package_output`` (verbose) chứ không qua ``packager``: lúc dừng/lúc lỗi
    là đúng lúc cần thấy zip có bao nhiêu checkpoint resume và bỏ sót file nào. Hai đường
    dùng chung một khoá theo đường dẫn zip nên không ghi đè nhau.
    """
    try:
        package_output(SAVE_DIR, ZIP_PATH, max_mb=150, keep_ckpts=True)
        print(f"\n>> Đã tạo {ZIP_PATH} ({reason}). TẢI VỀ trước khi phiên kết thúc.")
    except Exception as exc:
        print(
            f"\n[cảnh báo] không đóng gói được {ZIP_PATH}: {type(exc).__name__}: {exc}"
        )


print(f"[Run] notebook {NOTEBOOK_VERSION} | tập={DATASETS}")
for _name in DATASETS:
    _variants = tuple(VARIANTS_BY_DATASET.get(_name, ("train",)))
    _final = find_checkpoint(
        SAVE_DIR / _name, "GRKT-train", SPLIT_SEED
    ) or checkpoint_path(SAVE_DIR / _name, "GRKT-train", SPLIT_SEED)
    print(
        f"  {_name}: biến thể={_variants} | multiseed={_name in MULTI_SEED_DATASETS} | "
        f"{_final.name}={'CÓ (nạp, chỉ đánh giá)' if _final.exists() else 'KHÔNG (train từ đầu)'}"
    )

packager.start()
try:
    result = run_level2(
        DATASETS,
        save_root=SAVE_DIR,
        time_budget_min=TIME_BUDGET_MIN,
        ckpt_every_min=CKPT_EVERY_MIN,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        max_seq_len=MAX_SEQ_LEN,
        lr=LR,
        patience=PATIENCE,
        d_hidden=D_HIDDEN,
        k_hidden=K_HIDDEN,
        k_hop=K_HOP,
        alpha=ALPHA,
        graph_thresh=GRAPH_THRESH,
        cold_thresh=COLD_THRESH,
        gap_cap_min=GAP_CAP_MIN,
        max_users=MAX_USERS,
        variants=("train",),
        variants_by_dataset=VARIANTS_BY_DATASET,
        split_seed=SPLIT_SEED,
        run_baselines=RUN_BASELINES,
        baseline_ckpt_dir=(LEVEL1_DIR if USE_COMMITTED_CKPTS else None),
        baseline_epochs=BASELINE_EPOCHS,
        allow_retrain_baselines=not EVAL_ONLY,
        record_trajectory=RECORD_TRAJECTORY,
        run_calibration=RUN_CALIBRATION,
        run_multi_seed=RUN_MULTI_SEED,
        multi_seed_datasets=MULTI_SEED_DATASETS,
        seeds=SEEDS,
        resume_grkt_ckpt=RESUME_GRKT_CKPT,
        device=device,
        on_dataset_done=lambda _ds, _run: _package_now(f"xong tập {_ds}"),
    )
except TrainingPaused as exc:
    print("\n" + "!" * 78)
    print("DỪNG CHỦ ĐỘNG THEO NGÂN SÁCH — checkpoint đã ghi, không mất phần đã train.")
    print(" ", exc)
    print("!" * 78)
    _package_now("dừng theo ngân sách thời gian")
    raise
except Exception:
    _package_now("lỗi giữa chừng — gói checkpoint để phiên sau resume")
    raise
finally:
    # final_pack=False: hai nhánh except vừa đóng gói, còn nhánh thành công đóng gói ở cuối
    # cell (sau khi _rebuild_combined ghi lại level2_all_datasets.csv).
    packager.stop(final_pack=False)

combined = result["combined"]
print("\nSố dòng vừa sinh trong lần này:", len(combined))


def _rebuild_combined():
    """Đọc lại mọi <tập>/level2_metrics.csv trên đĩa để không xoá dòng của tập chạy trước."""
    order = list(DATASETS) + sorted(
        d.name
        for d in SAVE_DIR.iterdir()
        if d.is_dir() and (d / "level2_metrics.csv").exists() and d.name not in DATASETS
    )
    frames = []
    for name in order:
        path = SAVE_DIR / name / "level2_metrics.csv"
        if not path.exists():
            print(f"[cảnh báo] {name} chưa có level2_metrics.csv")
            continue
        t = pd.read_csv(path)
        t.insert(0, "dataset", name)
        frames.append(t)
    if not frames:
        print("[cảnh báo] không tìm thấy level2_metrics.csv nào -> giữ bảng cũ.")
        return combined
    all_df = pd.concat(frames, ignore_index=True)
    out_path = SAVE_DIR / "level2_all_datasets.csv"
    all_df.to_csv(out_path, index=False)
    print(
        f"Đã ghi {out_path} | {len(all_df)} dòng | tập={sorted(all_df['dataset'].unique())}"
    )
    return all_df


all_tables = _rebuild_combined()
_package_now("xong §3 — đã có bảng tổng hợp mọi tập")
print("Chạy §3.1 để kiểm tra sản phẩm bắt buộc; §12 đóng gói bản cuối kèm báo cáo/hình.")
all_tables

### 3.1 Kiểm tra sản phẩm bắt buộc (P0) và phạm vi tập con

Không raise ở đây: nếu thiếu mục nào thì vẫn đóng gói được phần đã có, nhưng bảng "THIẾU" phải
được xử lý (chạy bù) trước khi trích số vào báo cáo.


In [ ]:
REAL_CRITERIA = [
    "precedence",
    "precedence_gap",
    "stability_spearman",
    "stability_changed_frac",
    "forget_slope",
    "pearson_dt_dm",
]
lines = []


def _check(tag: str, ok: bool, detail: str) -> None:
    lines.append(f"[{'OK' if ok else 'THIẾU'}] {tag}: {detail}")


for name in DATASETS:
    d = SAVE_DIR / name
    mpath = d / "level2_metrics.csv"
    if not mpath.exists():
        _check(f"{name} metrics", False, f"không có {mpath}")
        continue
    t = pd.read_csv(mpath)
    _check(
        f"{name} metrics", True, f"{len(t)} dòng, model={sorted(t['model'].unique())}"
    )

    if name == "assist2009":
        zero = t[
            (t["model_type"] == "GRKT") & (t["graph"].astype(str).str.lower() == "zero")
        ]
        _check(
            "P0-2 ablation zero",
            not zero.empty,
            "có dòng graph=zero" if not zero.empty else "THIẾU GRKT-zero",
        )
        base = t[t["model_type"] != "GRKT"]
        bad = base[base["seed"].isna() | base["checkpoint_seed"].isna()]
        _check(
            "P0-4 đường nền",
            not base.empty and bad.empty,
            (
                "mọi baseline có seed + checkpoint_seed"
                if (not base.empty and bad.empty)
                else (
                    "không có baseline"
                    if base.empty
                    else f"thiếu seed ở {sorted(bad['model'].unique())}"
                )
            ),
        )

    tpath = d / "trajectory_metrics.csv"
    if not tpath.exists():
        _check(f"{name} quỹ đạo", False, f"không có {tpath}")
    else:
        tt = pd.read_csv(tpath)
        missing = [c for c in REAL_CRITERIA if c not in tt.columns]
        _check(
            f"{name} quỹ đạo",
            not missing,
            f"{len(tt)} dòng graph={sorted(tt['graph'].astype(str))}"
            + ("" if not missing else f" | thiếu cột {missing}"),
        )
        if name == "assist2009":
            _check(
                "P0-2 ablation trong quỹ đạo",
                len(tt) >= 2,
                f"{len(tt)} biến thể — cần >=2 để so train vs zero",
            )

    cpath = d / "calibration.csv"
    _check(f"{name} hiệu chỉnh", cpath.exists(), str(cpath))

    if name == "junyi":
        spath = d / "summary.json"
        if spath.exists():
            try:
                summ = json.loads(spath.read_text(encoding="utf-8"))
                users = int(summ.get("data_stats", {}).get("users", -1))
                _check(
                    "Junyi là tập con",
                    0 < users <= MAX_USERS * 1.05,
                    f"{users} người (cap {MAX_USERS})",
                )
                _check(
                    "Junyi chỉ 1 biến thể",
                    len(summ.get("graph_edges", {})) == 1,
                    f"graph_edges={list(summ.get('graph_edges', {}))}",
                )
            except Exception as exc:
                _check("Junyi summary", False, f"không đọc được: {exc}")
        else:
            _check("Junyi summary", False, f"không có {spath}")

ms_path = SAVE_DIR / "assist2009" / "multiseed_metrics.csv"
if "assist2009" in DATASETS:
    if not ms_path.exists():
        _check("P0-3 đa seed", False, f"không có {ms_path}")
    else:
        ms = pd.read_csv(ms_path)
        _check(
            "P0-3 đa seed",
            bool((ms["n_seed"] >= 3).all()),
            "n_seed="
            + ", ".join(f"{r.model}:{int(r.n_seed)}" for r in ms.itertuples()),
        )

for name in DATASETS:
    p = find_checkpoint(SAVE_DIR / name, "GRKT-train", SPLIT_SEED) or checkpoint_path(
        SAVE_DIR / name, "GRKT-train", SPLIT_SEED
    )
    _check(f"{name} checkpoint train", p.exists(), str(p))

print("\n".join(lines))
print(f"\n{sum(l.startswith('[OK]') for l in lines)}/{len(lines)} mục đạt.")
if any(l.startswith("[THIẾU]") for l in lines):
    print(
        "Mục [THIẾU] phải chạy bù trước khi đóng gói §12 hoặc trước khi trích số vào báo cáo."
    )

## 4. Provenance: cấu hình, môi trường, lịch sử train, resume

GAP F5 nêu đúng vấn đề: bảng kết quả không tự chứng minh được nó được sinh bởi cấu hình nào, train
bao nhiêu epoch, có resume từ checkpoint hay không. Cell này ghi:

- `environment.txt`: OS, Python, phiên bản thư viện, GPU;
- `run_sessions.json`: mỗi phiên một mục, kèm **history từng epoch** và `resumed_from`;
- `run_provenance.json`: bản rút gọn theo tập dữ liệu để đưa vào báo cáo;
- `figures/learning_curves_<tập>.png`: learning curve thật (val AUC + loss train/val) của mọi mô
  hình có history — trả lời GAP P1-5.

Với mô hình nạp từ checkpoint (resume), `history` để trống và `resumed_from` chỉ rõ file — không
bịa lại đường cong train.


In [ ]:
import json
import platform
import time

import matplotlib
import scipy
import sklearn

env_lines = [
    f"platform: {platform.platform()}",
    f"python: {sys.version}",
    f"torch: {torch.__version__}",
    f"numpy: {np.__version__}",
    f"pandas: {pd.__version__}",
    f"scipy: {scipy.__version__}",
    f"scikit-learn: {sklearn.__version__}",
    f"matplotlib: {matplotlib.__version__}",
    f"device: {device}",
    f"gpu: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}",
    f"cpu_count: {os.cpu_count()}",
    f"git_commit: {_head}",
    f"notebook: {NOTEBOOK_VERSION}",
]
(SAVE_DIR / "environment.txt").write_text("\n".join(env_lines), encoding="utf-8")
print("\n".join(env_lines))


def _nums(values) -> list:
    """Ép float và dừng ở phần tử hỏng/không hữu hạn — history có thể chứa None giữa chừng."""
    out = []
    for x in values or []:
        try:
            v = float(x)
        except (TypeError, ValueError):
            break
        if not np.isfinite(v):
            break
        out.append(v)
    return out


def _history_info(res: dict) -> dict:
    hist = res.get("history") or {}
    return {
        "history": {k: [round(v, 5) for v in _nums(v)] for k, v in hist.items()},
        "epochs_run": len(_nums(hist.get("val_auc"))),
        "seconds": float(res.get("train_seconds", 0.0) or 0.0),
        "resumed_from": res.get("resumed_from") or res.get("ckpt_source"),
        "from_checkpoint": bool(res.get("from_checkpoint", False)),
    }


def _plot_history(run: dict, ds: str) -> list:
    """Learning curve từ history THẬT; mô hình nạp checkpoint có history rỗng nên bị bỏ qua."""
    curves = []
    for res in run.get("baseline_results", []) + run.get("grkt_results", []):
        hist = res.get("history") or {}
        if len(_nums(hist.get("val_auc"))) < 2:
            continue
        curves.append((str(res["name"]), hist))
    if not curves:
        print(
            f"  [{ds}] không có history >= 2 epoch (toàn bộ nạp từ checkpoint) -> không vẽ learning curve"
        )
        return []

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    for label, hist in curves:
        auc = _nums(hist.get("val_auc"))
        if len(auc) >= 2:
            axes[0].plot(np.arange(1, len(auc) + 1), auc, marker="o", ms=3, label=label)
        for key, style, tag in (
            ("val_loss", "-", "val"),
            ("train_loss", "--", "train"),
        ):
            loss = _nums(hist.get(key))
            if len(loss) >= 2:
                axes[1].plot(
                    np.arange(1, len(loss) + 1),
                    loss,
                    style,
                    lw=1.2,
                    label=f"{label} {tag}",
                )
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("val AUC")
    axes[0].set_title(f"{ds} — validation AUC theo epoch")
    axes[0].legend(fontsize=7)
    axes[0].grid(alpha=0.3)
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("loss")
    axes[1].set_title(f"{ds} — loss (nét đứt = train)")
    axes[1].legend(fontsize=7, ncol=2)
    axes[1].grid(alpha=0.3)
    fig.tight_layout()
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    path = FIG_DIR / f"learning_curves_{ds}.png"
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"  [{ds}] learning curve -> {path}")
    return [str(path)]


provenance = {}
for name in DATASETS:
    run = result["runs"].get(name)
    if run is None:
        continue
    summ_path = Path(run["save_dir"]) / "summary.json"
    summ = {}
    if summ_path.exists():
        summ = json.loads(summ_path.read_text(encoding="utf-8"))
    models = []
    for res in run.get("baseline_results", []) + run.get("grkt_results", []):
        info = _history_info(res)
        info.update(
            {
                "model": res["name"],
                "graph": res.get("graph_type"),
                "test_auc": round(float(res["test_auc"]), 4),
                "test_acc": round(float(res["test_acc"]), 4),
                "test_loss": round(float(res["test_loss"]), 4),
                "seed": res.get("seed"),
                "checkpoint_seed": res.get("ckpt_seed"),
            }
        )
        models.append(info)
    provenance[name] = {
        "learning_curves": _plot_history(run, name),
        "save_dir": run["save_dir"],
        "canonical": summ.get("canonical"),
        "data_stats": summ.get("data_stats", {}),
        "graph_edges": summ.get("graph_edges", {}),
        "split_counts": {
            k: len(run["split"][f"{k}_users"]) for k in ("train", "val", "test")
        },
        "variants": list(run.get("trajectories", {})),
        "models": models,
    }

(SAVE_DIR / "run_provenance.json").write_text(
    json.dumps(provenance, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

_sessions = json.loads((SAVE_DIR / "run_sessions.json").read_text(encoding="utf-8"))
_sessions[-1].update(
    {
        "status": "completed",
        "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "git_commit": _head,
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "datasets": provenance,
    }
)
(SAVE_DIR / "run_sessions.json").write_text(
    json.dumps(_sessions, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print(f"\nĐã ghi {SAVE_DIR/'run_provenance.json'} và cập nhật run_sessions.json.")
for name, info in provenance.items():
    print(
        f"[{name}] split={info['split_counts']} | graph={info['graph_edges']} | "
        f"{len(info['models'])} dòng mô hình"
    )

## 5. So sánh DKT / GKT / GRKT và liên kết Cấp độ 1

Hai mục:

1. **Bảng chính** trên cùng split, cùng chỉ số (AUC / Accuracy / NLL), kèm `seed` (seed đã train
   ra trọng số) và `checkpoint_seed` (seed của checkpoint đường nền). Ô trống nghĩa là **không
   xác định được**, không phải 42.
2. **Liên kết Cấp độ 1**, gồm ba kiểm tra tách bạch:
   - tập người học train/val/test có trùng `outputs/level1/split.json` hay không;
   - AUC đường nền train lại seed 42 lệch bao nhiêu so với `level1_metrics.csv`;
   - (tuỳ chọn) đo riêng checkpoint Cấp độ 1 đã commit để kết luận về GAP F2.


In [ ]:
from IPython.display import display

levels = pd.read_csv(SAVE_DIR / "level2_all_datasets.csv")
show_cols = [
    "dataset",
    "model",
    "graph",
    "auc",
    "acc",
    "loss",
    "val_auc",
    "n_test",
    "seed",
    "checkpoint_seed",
    "seconds",
]
print("=== BẢNG CHÍNH (cùng split theo người học, cùng chỉ số) ===")
for ds in levels["dataset"].unique():
    sub = levels[levels["dataset"] == ds]
    print(f"\n[{ds}]")
    display(sub[[c for c in show_cols if c in sub.columns]])

fig, axes = plt.subplots(
    1,
    len(levels["dataset"].unique()),
    figsize=(5.2 * len(levels["dataset"].unique()), 3.8),
    squeeze=False,
)
for ax, ds in zip(axes[0], levels["dataset"].unique()):
    sub = levels[levels["dataset"] == ds]
    labels = [
        f"{m}\n{g}" if str(g) != "N/A" else m
        for m, g in zip(sub["model"], sub["graph"])
    ]
    colors = ["#9aa5b1" if str(g) == "N/A" else "#4878a6" for g in sub["graph"]]
    bars = ax.bar(labels, sub["auc"], color=colors)
    ax.set_ylim(
        max(0.5, float(sub["auc"].min()) - 0.04),
        min(1.0, float(sub["auc"].max()) + 0.04),
    )
    ax.set_ylabel("Test AUC")
    ax.set_title(ds)
    ax.tick_params(axis="x", labelsize=7)
    for b, v in zip(bars, sub["auc"]):
        ax.text(
            b.get_x() + b.get_width() / 2,
            v + 0.002,
            f"{v:.4f}",
            ha="center",
            fontsize=7,
        )
fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


# ---------------------------------------------------------------- liên kết Cấp độ 1
def _load_json_maybe_lfs(path: Path):
    """Trả (obj, status); status='git-lfs-pointer' nếu file chưa được fetch."""
    if not path.exists():
        return None, "missing"
    text = path.read_text(encoding="utf-8", errors="replace")
    if (
        text.lstrip().startswith("version https://git-lfs")
        or len(text) < 200
        and "oid sha256" in text
    ):
        return None, "git-lfs-pointer"
    try:
        return json.loads(text), "ok"
    except json.JSONDecodeError as exc:
        return None, f"invalid-json: {exc}"


def _read_level1_table(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path, encoding="utf-8-sig")
    return df.rename(
        columns={
            "Mô hình": "model",
            "Test AUC": "ref_auc",
            "Test Accuracy": "ref_acc",
            "Test Loss (NLL)": "ref_loss",
        }
    )


level1_ref = _read_level1_table(LEVEL1_DIR / "level1_metrics.csv")
level1_multiseed = _read_level1_table(LEVEL1_DIR / "multiseed_metrics.csv")

l2_split, l2_status = _load_json_maybe_lfs(SAVE_DIR / "assist2009" / "split.json")
l1_split, l1_status = _load_json_maybe_lfs(LEVEL1_DIR / "split.json")

split_rows = []
same_split = None
if l2_split and l1_split:
    same_split = True
    for key in ("train", "val", "test"):
        a = set(int(x) for x in l1_split.get(f"{key}_users", []))
        b = set(int(x) for x in l2_split.get(f"{key}_users", []))
        same_split = same_split and (a == b)
        split_rows.append(
            {
                "split": key,
                "level1": len(a),
                "level2": len(b),
                "identical": bool(a == b),
                "only_level1": len(a - b),
                "only_level2": len(b - a),
            }
        )
    print("\n=== ĐỐI CHIẾU TẬP NGƯỜI HỌC VỚI CẤP ĐỘ 1 ===")
    display(pd.DataFrame(split_rows))
    print(
        "Kết luận:",
        (
            "TRÙNG HOÀN TOÀN"
            if same_split
            else "KHÁC — không dùng số liệu hai cấp để so trực tiếp"
        ),
    )
else:
    print(
        f"\n[đối chiếu split] bỏ qua: level1 split={l1_status}, level2 split={l2_status}. "
        "Cùng seed 42 + cùng hàm load_kt_data vẫn bảo đảm split, nhưng không có bằng chứng file."
    )

link_rows = []
if level1_ref is not None and "assist2009" in set(levels["dataset"]):
    print("\n=== AUC ĐƯỜNG NỀN SO VỚI BẢNG CẤP ĐỘ 1 (seed 42) ===")
    same_weights = bool(USE_COMMITTED_CKPTS)
    tol = 0.005 if (MAX_SEQ_LEN == 200 and same_weights) else 0.02
    for model in ("DKT", "GKT-Transition", "GKT-Zero"):
        row = levels[(levels["dataset"] == "assist2009") & (levels["model"] == model)]
        ref = level1_ref[level1_ref["model"] == model]
        if not len(row) or not len(ref):
            continue
        a2 = float(row["auc"].iloc[0])
        a1 = float(ref["ref_auc"].iloc[0])
        ck_seed = row["checkpoint_seed"].iloc[0]
        known_seed = bool(pd.notna(ck_seed))
        status = "OK" if abs(a2 - a1) <= tol else "KHÁC > tol"
        link_rows.append(
            {
                "model": model,
                "level1_auc": a1,
                "level2_auc": a2,
                "delta": round(a2 - a1, 4),
                "tolerance": tol,
                "checkpoint_seed": None if not known_seed else int(ck_seed),
                "same_weights": same_weights,
                "status": status,
            }
        )
        print(
            f"{model:16s} | C1={a1:.4f} | C2={a2:.4f} | Δ={a2 - a1:+.4f} | "
            f"checkpoint_seed={'?' if not known_seed else int(ck_seed)} | {status}"
        )
    print(
        "Diễn giải: USE_COMMITTED_CKPTS=False nên đường nền được TRAIN LẠI seed 42. Δ nhỏ là "
        "kiểm tra split/cửa sổ, không chứng minh trọng số giống hệt Cấp độ 1."
    )
else:
    print(
        "\n[đối chiếu AUC] không có level1_metrics.csv hoặc không chạy assist2009 -> bỏ qua."
    )

linkage = {
    "notebook": NOTEBOOK_VERSION,
    "git_commit": _head,
    "level1_dir": str(LEVEL1_DIR),
    "level1_split_status": l1_status,
    "level2_split_status": l2_status,
    "same_split": same_split,
    "split_rows": split_rows,
    "same_weights": bool(USE_COMMITTED_CKPTS),
    "max_seq_len": int(MAX_SEQ_LEN),
    "split_seed": int(SPLIT_SEED),
    "baseline_comparison": link_rows,
    "level1_reference_table": (
        None if level1_ref is None else level1_ref.to_dict(orient="records")
    ),
    "level1_multiseed": (
        None if level1_multiseed is None else level1_multiseed.to_dict(orient="records")
    ),
}
(SAVE_DIR / "diagnostics").mkdir(parents=True, exist_ok=True)
(SAVE_DIR / "diagnostics" / "level1_linkage.json").write_text(
    json.dumps(linkage, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print(f"\nĐã ghi {SAVE_DIR/'diagnostics'/'level1_linkage.json'}")

### 5.2 Đo riêng checkpoint Cấp độ 1 đã commit (chẩn đoán GAP F2)

`outputs/level1/*.pt` được commit ngày 16/10/2026, **trước** bản sửa đặt hậu tố `-seed{s}` cho
seed khác 42 (01/11). Vì vậy không có bằng chứng chúng là seed 42 — rất có thể chúng là trọng số
của seed chạy cuối trong vòng đa seed. Cell này chỉ **đo** các checkpoint đó (không dùng cho bảng
chính, không train lại) và so với:

- bảng Cấp độ 1 (`level1_metrics.csv`, seed 42);
- đường nền vừa train lại trong Cấp độ 2 (seed 42).

Nếu checkpoint cũ không tái lập được bảng Cấp độ 1, kết luận là F2 đúng và bảng chính phải dùng
đường nền train lại — đúng như cấu hình mặc định của notebook.


In [ ]:
legacy = None
legacy_error = None
if (
    EVAL_COMMITTED_CKPTS
    and "assist2009" in DATASETS
    and find_checkpoint(LEVEL1_DIR, "DKT", 42) is not None
):
    try:
        from level1.pipeline import load_kt_data
        from level2.baselines import level1_baseline

        summ_path = SAVE_DIR / "assist2009" / "summary.json"
        canonical = json.loads(summ_path.read_text(encoding="utf-8"))["canonical"]
        shared_l1 = load_kt_data(
            canonical,
            batch_size=BATCH_SIZE,
            graph_type="Transition",
            model_type="GKT",
            train_ratio=0.6,
            val_ratio=0.2,
            seed=SPLIT_SEED,
            max_seq_len=MAX_SEQ_LEN,
            min_seq_len=get_spec("assist2009").min_interactions,
            use_cuda=(device.type == "cuda"),
        )
        got = level1_baseline(
            shared_l1,
            specs=BASELINE_SPECS,
            ckpt_dir=LEVEL1_DIR,
            epochs=BASELINE_EPOCHS,
            batch_size=BATCH_SIZE,
            max_seq_len=MAX_SEQ_LEN,
            min_seq_len=get_spec("assist2009").min_interactions,
            train_ratio=0.6,
            val_ratio=0.2,
            seed=SPLIT_SEED,
            device=device,
            allow_retrain=False,
            collect_probs=False,
            save_dir=SAVE_DIR / "diagnostics" / "legacy_ckpt",
            verbose=True,
        )
        legacy = pd.DataFrame(
            [
                {
                    "model": r["name"],
                    "legacy_auc": round(float(r["test_auc"]), 4),
                    "legacy_acc": round(float(r["test_acc"]), 4),
                    "legacy_loss": round(float(r["test_loss"]), 4),
                    "checkpoint_seed": r.get("ckpt_seed"),
                    "ckpt_source": r.get("ckpt_source"),
                }
                for r in got["results"]
            ]
        )
        out = SAVE_DIR / "diagnostics" / "legacy_ckpt_eval.csv"
        legacy.to_csv(out, index=False)
        print(f"\nĐã ghi {out}")
        display(legacy)
        if level1_ref is not None:
            merged = legacy.merge(
                level1_ref[["model", "ref_auc"]], on="model", how="left"
            )
            merged["delta_vs_level1"] = (
                merged["legacy_auc"] - merged["ref_auc"]
            ).round(4)
            merged["reproduces_level1"] = merged["delta_vs_level1"].abs() <= 0.005
            display(
                merged[
                    [
                        "model",
                        "legacy_auc",
                        "ref_auc",
                        "delta_vs_level1",
                        "reproduces_level1",
                    ]
                ]
            )
            verdict = (
                "Checkpoint cũ TÁI LẬP được bảng C1 -> có thể coi là seed 42."
                if bool(merged["reproduces_level1"].all())
                else "Checkpoint cũ KHÔNG tái lập bảng C1 -> khớp GAP F2; dùng đường nền train lại."
            )
            print("Kết luận:", verdict)
        else:
            verdict = "Không có level1_metrics.csv -> chỉ ghi số đo, không kết luận."
    except Exception as exc:
        legacy_error = f"{type(exc).__name__}: {exc}"
        verdict = f"Không đo được checkpoint cũ: {legacy_error}"
        print("[cảnh báo]", verdict)
else:
    verdict = "EVAL_COMMITTED_CKPTS=False hoặc không có checkpoint -> bỏ qua."
    print(verdict)

linkage["legacy_ckpt"] = {
    "evaluated": legacy is not None,
    "error": legacy_error,
    "verdict": verdict,
    "rows": None if legacy is None else legacy.to_dict(orient="records"),
}
(SAVE_DIR / "diagnostics" / "level1_linkage.json").write_text(
    json.dumps(linkage, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

## 6. Quỹ đạo mastery m(u, c, t) và tiêu chí hợp lý

Đây là nhiệm vụ bắt buộc của Cấp độ 2. Đọc bảng theo hai nhóm:

- **tiêu chí thật** — `precedence_gap`, `stability_spearman`, `forget_slope`, `pearson_dt_dm`:
  có thể sai, dùng làm bằng chứng;
- **kiểm chứng cài đặt** — `consistency_strengthen`, `consistency_spread`, `locality`: bị cấu
  trúc mô hình ép ≡ 1. `≠ 1` mới là lỗi hook, còn `= 1` **không** chứng minh đồ thị có ích.

So `train` với `zero`: chênh lệch `abs_delta_retrieval` và `abs_delta_learn_forget` là phần thay
đổi quỹ đạo do đồ thị gây ra.


In [ ]:
REAL_COLS = [
    "graph",
    "users",
    "steps",
    "precedence",
    "precedence_gap",
    "stability_spearman",
    "stability_changed_frac",
    "forget_slope",
    "pearson_dt_dm",
]
INSTALL_COLS = ["graph", "consistency_strengthen", "consistency_spread", "locality"]
STAGE_COLS = [
    "graph",
    "abs_delta_retrieval",
    "abs_delta_strengthen",
    "abs_delta_learn_forget",
]

for name, run in result["runs"].items():
    tt = run.get("trajectory_table")
    if tt is None or len(run.get("trajectories", {})) == 0:
        print(f"[{name}] không có quỹ đạo (RECORD_TRAJECTORY=False?)")
        continue
    kind0 = sorted(run["trajectories"])[0]
    print("=" * 78)
    print(
        f"[{name}] QUỸ ĐẠO MASTERY — {len(run['trajectories'][kind0]['users'])} người học thuộc test"
    )
    print("=" * 78)
    print("\n-- TIÊU CHÍ THẬT (dùng làm bằng chứng) --")
    display(tt[[c for c in REAL_COLS if c in tt.columns]])
    print("\n-- KIỂM CHỨNG CÀI ĐẶT (kỳ vọng = 1,0; KHÔNG phải bằng chứng đồ thị) --")
    display(tt[[c for c in INSTALL_COLS if c in tt.columns]])
    print("\n-- PHÂN TÁCH GIAI ĐOẠN (|Δm| trung bình) --")
    display(tt[[c for c in STAGE_COLS if c in tt.columns]])

    grkt = [r for r in run.get("grkt_results", []) if r.get("model_type") == "GRKT"]
    if len(grkt) > 1:
        cmp = pd.DataFrame(
            [
                {
                    "graph": r.get("graph_type"),
                    "test_auc": round(float(r["test_auc"]), 4),
                    "edges_rel": int(r.get("edges_rel", 0)),
                    "edges_pre": int(r.get("edges_pre", 0)),
                }
                for r in grkt
            ]
        )
        display(cmp)
        if {"train", "zero"}.issubset(set(cmp["graph"])):
            d_auc = float(cmp.loc[cmp["graph"] == "train", "test_auc"].iloc[0]) - float(
                cmp.loc[cmp["graph"] == "zero", "test_auc"].iloc[0]
            )
            print(
                f"ΔAUC(train − zero) = {d_auc:+.4f} -> "
                + (
                    "có lợi ích từ đồ thị"
                    if d_auc > 0
                    else "KHÔNG thấy lợi ích của đồ thị"
                )
            )
    else:
        print(
            f"[{name}] chỉ 1 biến thể -> không có so sánh chéo; ghi rõ giới hạn khi báo cáo."
        )

    for kind in sorted(run.get("trajectories", {})):
        png = Path(run["save_dir"]) / "trajectories" / f"heatmap_{kind}.png"
        if png.exists():
            import matplotlib.image as mpimg

            plt.figure(figsize=(7, 4))
            plt.imshow(mpimg.imread(png))
            plt.axis("off")
            plt.title(f"{name} / {kind}: mastery theo khái niệm × thời gian")
            plt.tight_layout()
            plt.show()

## 7. Hiệu chỉnh xác suất (ECE / Brier)

AUC chỉ đo thứ hạng. Để biết xác suất có dùng được cho dashboard hay không, cần calibration.
`RUN_CALIBRATION=True` nên bảng này có cho mọi mô hình; đây là phân tích nâng cao thứ hai bên
cạnh quỹ đạo.


In [ ]:
for name, run in result["runs"].items():
    cal = run.get("calibration")
    if cal is None or len(cal) == 0:
        print(f"[{name}] bỏ qua (RUN_CALIBRATION=False).")
        continue
    print("=" * 72)
    print(f"[{name}] HIỆU CHỈNH (sắp theo ECE tăng dần)")
    print("=" * 72)
    display(cal)
    png = Path(run["save_dir"]) / "reliability.png"
    if png.exists():
        import matplotlib.image as mpimg

        plt.figure(figsize=(6.4, 5.2))
        plt.imshow(mpimg.imread(png))
        plt.axis("off")
        plt.tight_layout()
        plt.show()

## 8. Đánh giá đa seed (assist2009)

`YEU_CAU.md` §7 yêu cầu mean ± std trên ≥3 seed. Notebook đa seed chỉ bật cho assist2009 (reuse
checkpoint seed 42, train thêm 123 và 456); Junyi chạy 1 seed và điều đó được ghi rõ — không có
bảng "mean ± std" giả cho Junyi.


In [ ]:
for name, run in result["runs"].items():
    if "multiseed" not in run:
        print(
            f"[{name}] không chạy đa seed (MULTI_SEED_DATASETS={MULTI_SEED_DATASETS})."
        )
        continue
    print("=" * 72)
    print(f"[{name}] ĐA SEED GRKT (split cố định seed={SPLIT_SEED})")
    print("=" * 72)
    display(run["multiseed_raw"])
    display(run["multiseed"])

    ms = run["multiseed"]
    fig, ax = plt.subplots(figsize=(6.2, 3.4))
    ax.bar(ms["model"], ms["auc_mean"], yerr=ms["auc_std"], capsize=5, color="#4878a6")
    ax.set_ylim(
        max(0.5, float((ms["auc_mean"] - ms["auc_std"]).min()) - 0.02),
        min(1.0, float((ms["auc_mean"] + ms["auc_std"]).max()) + 0.02),
    )
    ax.set_ylabel("Test AUC (mean ± std)")
    ax.set_title(f"GRKT trên {name} — {int(ms['n_seed'].max())} seed")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"multiseed_{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9. Dashboard người học Cấp độ 2 (YEU_CAU §8, GAP F7)

Bản cũ thiếu hẳn mục này. Dashboard ở đây **không train lại** và **không đọc lại checkpoint
bằng pickle tuỳ ý**:

- mastery lấy từ `trajectories/traj_<biến-thể>.npz` (đã ghi ở §3, đọc bằng
  `load_trajectories`, không pickle);
- dự đoán từng bước lấy từ model GRKT còn trong bộ nhớ của lần chạy này, qua
  `predict_grkt` — đúng hàm dùng cho ECE/Brier;
- mỗi dòng là một bước `t ≥ 1`: xác suất dự đoán cho bước đó, nhãn thật, mastery **trước**
  bước đó của (các) khái niệm được làm, mastery sau lan truyền đồ thị, **bằng chứng lịch sử**
  (tối đa 3 lần làm đúng/sai khái niệm đó trước bước `t`) và **mastery 1-hop trên đồ thị**
  (trung bình các khái niệm láng giềng trước bước `t`). Hai cột cuối là phần "bằng chứng cho
  dự đoán" mà `YEU_CAU.md` §8 yêu cầu, không phải suy diễn thêm.

Đối chiếu căn chỉnh: GRKT trả `scores[:, 1:]` và `labels = corrs[:, 1:]`, nên `p[bước t]` ứng
với `m_pre[t]` và `correct[t]` — cell tự kiểm tra độ dài trước khi ghép, lệch thì bỏ dự đoán
thay vì ghép sai.

Xuất ra: `dashboard/learner_dashboard.json`, `dashboard/learner_dashboard_summary.csv`,
`dashboard/learner_dashboard_steps.csv`, `figures/dashboard_<tập>.png`, kèm dropdown
ipywidgets (nếu môi trường có) và bảng in thay thế nếu không.


In [ ]:
from level2.grkt_api import predict_grkt
from level2.trajectory import load_trajectories
from sklearn.metrics import roc_auc_score

DASH_DIR = SAVE_DIR / "dashboard"
DASH_DIR.mkdir(parents=True, exist_ok=True)
DASH_JSON_USERS = min(int(DASHBOARD_MAX_LEARNERS), 5)


def _traj_from_disk(run: dict, kind: str):
    """Ưu tiên npz trên đĩa để dashboard tái lập được ở phiên resume; lùi về bộ nhớ."""
    p = Path(run["save_dir"]) / "trajectories" / f"traj_{kind}.npz"
    if p.exists():
        try:
            return load_trajectories(p)
        except Exception as exc:
            print(f"[cảnh báo] không đọc được {p}: {type(exc).__name__}: {exc}")
    return (run.get("trajectories") or {}).get(kind)


def _grkt_model(run: dict, kind: str):
    for r in run.get("grkt_results", []):
        if (
            r.get("model_type") == "GRKT"
            and r.get("graph_type") == kind
            and r.get("model") is not None
        ):
            return r["model"], r["name"]
    return None, None


dashboard = {
    "notebook": NOTEBOOK_VERSION,
    "git_commit": _head,
    "max_learners": int(DASHBOARD_MAX_LEARNERS),
    "note": (
        "Junyi là TẬP CON — dashboard chỉ minh hoạ vài người học trong tập test, "
        "không đại diện thống kê cho toàn bộ Junyi."
    ),
    "datasets": {},
}
summary_rows, step_rows = [], []

for name, run in result["runs"].items():
    kinds = sorted(run.get("trajectories") or {})
    if not kinds:
        print(
            f"[{name}] không có quỹ đạo (RECORD_TRAJECTORY=False?) -> bỏ qua dashboard."
        )
        continue
    test_seqs = ((run.get("splits") or {}).get("test") or {}).get("sequences") or []
    by_user = {s.user_id: s for s in test_seqs}
    skill_lookup = getattr(run.get("grkt_data"), "skill_lookup", None) or {}
    dashboard["datasets"][name] = {"kinds": {}}

    for kind in kinds:
        traj = _traj_from_disk(run, kind)
        if not traj or not traj.get("users"):
            print(f"[{name}/{kind}] không đọc được quỹ đạo -> bỏ qua.")
            continue
        model, model_name = _grkt_model(run, kind)
        if model is None:
            print(
                f"[{name}/{kind}] không có model trong bộ nhớ -> chỉ xuất mastery, không có dự đoán."
            )

        entries = []
        for u in traj["users"][: int(DASHBOARD_MAX_LEARNERS)]:
            uid = u["user_id"]
            corrs = np.asarray(u["corrs"]).astype(int)
            times = np.asarray(u["times"]).astype(int)
            knows = np.asarray(u["knows"])
            m_pre = np.asarray(u["m_pre"], dtype=float)
            m_tilde = np.asarray(u["m_tilde"], dtype=float)
            n = len(corrs)

            pred, align_note = None, "không dự đoán"
            seq = by_user.get(uid)
            if model is not None and seq is not None and n >= 2:
                try:
                    p, _y = predict_grkt(model, [seq.as_grkt_items()], device=device)
                    if p.size == n - 1:
                        pred, align_note = (
                            np.clip(p.astype(float), 0.0, 1.0),
                            "khớp S-1",
                        )
                    else:
                        align_note = f"lệch độ dài: scores={p.size}, S-1={n - 1} -> bỏ"
                except Exception as exc:
                    align_note = f"lỗi: {type(exc).__name__}: {exc}"

            steps, m_answered, nb_mastery = [], [], []
            recent = {}
            for k in [int(x) for x in knows[0] if int(x) > 0] if n else []:
                recent[k] = [(0, int(corrs[0]))]
            adj = run.get("adj")
            for i in range(1, n):
                ans = [int(k) for k in knows[i] if int(k) > 0]
                names = [str(skill_lookup.get(k - 1, f"k{k - 1}")) for k in ans]
                m_vals = [float(m_pre[i, k]) for k in ans]
                t_vals = [float(m_tilde[i, k]) for k in ans]
                m_here = float(np.max(m_vals)) if m_vals else float("nan")
                m_answered.append(m_here)

                # bằng chứng cho dự đoán: các lần làm ĐÚNG khái niệm này TRƯỚC bước i
                hist = sorted({(t, c) for k in ans for t, c in recent.get(k, [])})[-3:]
                hist_txt = "; ".join(f"t{t}:{'đúng' if c else 'sai'}" for t, c in hist)
                nb_val = None
                if adj is not None and ans:
                    nb = set()
                    for a in ans:
                        nb.update(int(c) for c in np.nonzero(np.asarray(adj)[a])[0])
                    nb -= set(ans)
                    nb.discard(0)
                    if nb:
                        nb_val = float(np.mean([m_pre[i, c] for c in nb]))
                        nb_mastery.append(nb_val)

                steps.append(
                    {
                        "dataset": name,
                        "graph": kind,
                        "user_id": str(uid),
                        "step": i,
                        "minute": int(times[i]),
                        "skills": "|".join(names),
                        "history_same_skill": hist_txt,
                        "n_prior_same_skill": len(hist),
                        "mastery_pre": None if np.isnan(m_here) else round(m_here, 4),
                        "mastery_tilde": (
                            None if not t_vals else round(float(np.max(t_vals)), 4)
                        ),
                        "neighbor_mastery": (
                            None if nb_val is None else round(nb_val, 4)
                        ),
                        "pred": None if pred is None else round(float(pred[i - 1]), 4),
                        "correct": int(corrs[i]),
                    }
                )
                for k in ans:
                    recent.setdefault(k, []).append((i, int(corrs[i])))

            lab = corrs[1:].astype(int)
            if pred is not None and len(pred) == len(lab):
                brier = float(np.mean((pred - lab) ** 2))
                auc = (
                    float(roc_auc_score(lab, pred))
                    if 0 < int(lab.sum()) < len(lab)
                    else None
                )
                corr_p = (
                    float(np.corrcoef(pred, lab)[0, 1])
                    if float(lab.std()) > 0
                    else None
                )
                mean_pred = float(np.mean(pred))
            else:
                brier = auc = corr_p = mean_pred = None

            k = max(1, len(m_answered) // 3)
            first = float(np.nanmean(m_answered[:k])) if m_answered else float("nan")
            last = float(np.nanmean(m_answered[-k:])) if m_answered else float("nan")
            n_ev = sum(1 for s in steps if s["n_prior_same_skill"] > 0)
            summary = {
                "dataset": name,
                "graph": kind,
                "user_id": str(uid),
                "steps": int(len(steps)),
                "acc": round(float(lab.mean()), 4) if len(lab) else None,
                "mean_pred": None if mean_pred is None else round(mean_pred, 4),
                "brier": None if brier is None else round(brier, 4),
                "auc": None if auc is None else round(auc, 4),
                "corr_pred_correct": None if corr_p is None else round(corr_p, 4),
                "mastery_first": None if np.isnan(first) else round(first, 4),
                "mastery_last": None if np.isnan(last) else round(last, 4),
                "mastery_delta": (
                    None
                    if (np.isnan(first) or np.isnan(last))
                    else round(last - first, 4)
                ),
                "mean_neighbor_mastery": (
                    round(float(np.mean(nb_mastery)), 4) if nb_mastery else None
                ),
                "evidence_rate": round(n_ev / len(steps), 4) if steps else None,
                "alignment": align_note,
                "model": model_name,
            }
            summary_rows.append(summary)
            step_rows.extend(steps)
            entries.append({"summary": summary, "steps": steps})

        if model is not None:
            model.cpu()
        for j, e in enumerate(entries):
            if j >= DASH_JSON_USERS:
                e["steps"] = e["steps"][:0]
                e["steps_note"] = "cắt bớt trong JSON; xem learner_dashboard_steps.csv"
        dashboard["datasets"][name]["kinds"][kind] = {
            "model": model_name,
            "n_learners": len(entries),
            "learners": entries,
        }
        print(
            f"[{name}/{kind}] dashboard: {len(entries)} người học, "
            f"model={model_name or 'không có'}, căn chỉnh dự đoán: {entries[0]['summary']['alignment'] if entries else 'n/a'}"
        )

    # hình minh hoạ: tối đa 4 người học của biến thể đầu tiên
    sub = pd.DataFrame(
        [r for r in step_rows if r["dataset"] == name and r["graph"] == kinds[0]]
    )
    if not sub.empty:
        users4 = list(dict.fromkeys(sub["user_id"]))[:4]
        fig, axes = plt.subplots(
            len(users4), 1, figsize=(9.2, 2.4 * len(users4)), squeeze=False
        )
        for ax, uid in zip(axes[:, 0], users4):
            g = sub[sub["user_id"] == uid]
            if g["pred"].notna().any():
                ax.plot(
                    g["step"], g["pred"], color="#4878a6", lw=1.3, label="p(dự đoán)"
                )
                hit = g["correct"] == 1
                ax.scatter(
                    g.loc[hit, "step"],
                    g.loc[hit, "pred"],
                    s=9,
                    color="#2e7d32",
                    label="đúng",
                )
                ax.scatter(
                    g.loc[~hit, "step"],
                    g.loc[~hit, "pred"],
                    s=9,
                    color="#c62828",
                    label="sai",
                )
            else:
                ax.text(
                    0.02,
                    0.5,
                    "không có dự đoán (thiếu model trong bộ nhớ)",
                    transform=ax.transAxes,
                    fontsize=7,
                    color="#777",
                )
            ax.set_ylim(-0.02, 1.02)
            ax.set_ylabel("p", fontsize=8)
            ax.tick_params(labelsize=7)
            ax2 = ax.twinx()
            ax2.plot(
                g["step"],
                g["mastery_pre"],
                color="#f9a825",
                lw=1.1,
                ls="--",
                label="mastery",
            )
            ax2.set_ylim(-0.02, 1.02)
            ax2.set_ylabel("m", fontsize=8, color="#f9a825")
            ax2.tick_params(labelsize=7)
            ax.set_title(f"{name}/{kinds[0]} · user {uid} · {len(g)} bước", fontsize=9)
            ax.legend(fontsize=6, loc="lower left")
        fig.tight_layout()
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f"dashboard_{name}.png", dpi=140, bbox_inches="tight")
        plt.show()

summary_df = pd.DataFrame(summary_rows)
steps_df = pd.DataFrame(step_rows)
if not summary_df.empty:
    summary_df.to_csv(DASH_DIR / "learner_dashboard_summary.csv", index=False)
if not steps_df.empty:
    steps_df.to_csv(DASH_DIR / "learner_dashboard_steps.csv", index=False)
(DASH_DIR / "learner_dashboard.json").write_text(
    json.dumps(dashboard, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print(
    f"\nĐã ghi {DASH_DIR/'learner_dashboard_summary.csv'} ({len(summary_df)} người học), "
    f"{DASH_DIR/'learner_dashboard_steps.csv'} ({len(steps_df)} bước), "
    f"{DASH_DIR/'learner_dashboard.json'}"
)
if not summary_df.empty:
    display(summary_df)
else:
    print(
        "Không có người học nào trong dashboard — kiểm tra RECORD_TRAJECTORY và split test."
    )

# ---------------------------------------------------------------- dropdown (tuỳ chọn)
if not summary_rows:
    print("Không có người học -> bỏ dropdown.")
else:
    try:
        import ipywidgets as widgets

        _opts = [
            (
                f"{r['dataset']}/{r['graph']} · user {r['user_id']}",
                (r["dataset"], r["graph"], r["user_id"]),
            )
            for r in summary_rows
        ]
        picker = widgets.Dropdown(
            options=_opts, description="Người học", layout=widgets.Layout(width="420px")
        )
        out_area = widgets.Output()

        def _show(_change=None):
            ds, kind, uid = picker.value
            g = steps_df[
                (steps_df["dataset"] == ds)
                & (steps_df["graph"] == kind)
                & (steps_df["user_id"] == uid)
            ]
            with out_area:
                out_area.clear_output()
                display(g.reset_index(drop=True))

        picker.observe(_show, names="value")
        display(picker, out_area)
        _show()
    except Exception as exc:
        print(
            f"[ipywidgets không dùng được: {type(exc).__name__}: {exc}] -> in bảng thay thế."
        )
        if not steps_df.empty:
            display(steps_df.head(30))

## 10. Kiểm kê sản phẩm và trạng thái checkpoint

Trước khi đóng gói, liệt kê mọi file đã sinh **tới thời điểm này** (kèm dung lượng) và đọc
**fingerprint** của từng checkpoint `GRKT-*.pt` để trả lời ba câu: checkpoint train trên bao nhiêu
người, split nào, có khớp `run_config.json` không. Đây là phần chứng minh resume không trộn cấu
hình (GAP F5), và cũng là chỗ phát hiện sớm checkpoint còn sót của lần chạy Junyi toàn bộ (F3).
`report_summary.md`/`README_pack.txt` sinh ở §11–§12 nên chưa có trong bảng này.


In [ ]:
import re

TAB_DIR.mkdir(parents=True, exist_ok=True)
rows = []
for p in sorted(SAVE_DIR.rglob("*")):
    if not p.is_file():
        continue
    rel = p.relative_to(SAVE_DIR).as_posix()
    parts = rel.split("/")
    rows.append(
        {
            "path": rel,
            "group": parts[0] if len(parts) > 1 else ".",
            "mb": round(p.stat().st_size / 1048576, 3),
        }
    )
artifacts = pd.DataFrame(rows)
artifacts.to_csv(TAB_DIR / "artifacts.csv", index=False)
print(f"{len(artifacts)} file | {artifacts['mb'].sum():.1f} MB trong {SAVE_DIR}")
if not artifacts.empty:
    display(
        artifacts.groupby("group")
        .agg(files=("path", "count"), mb=("mb", "sum"))
        .round(3)
    )


def _ckpt_info(path: Path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"
    if not isinstance(payload, dict):
        return None, "payload không phải dict"
    return payload, None


ck_rows = []
for d in sorted(p for p in SAVE_DIR.iterdir() if p.is_dir()):
    for p in sorted(d.glob("GRKT-*.pt")) + sorted(d.glob("ckpt_*.pt")):
        payload, err = _ckpt_info(p)
        fp = (payload or {}).get("fingerprint") or {}
        ck_rows.append(
            {
                "dataset": d.name,
                "file": p.name,
                "mb": round(p.stat().st_size / 1048576, 2),
                "format": (payload or {}).get("format"),
                "seed": fp.get("seed"),
                "n_train": fp.get("n_train"),
                "n_test": fp.get("n_test"),
                "n_steps_train": fp.get("n_steps_train"),
                "batch_size": fp.get("batch_size"),
                "d_hidden": fp.get("d_hidden"),
                "edges_rel": fp.get("edges_rel"),
                "edges_pre": fp.get("edges_pre"),
                "epochs_cap": (payload or {}).get("epochs"),
                "error": err,
            }
        )
ck_df = pd.DataFrame(ck_rows)
if not ck_df.empty:
    ck_df.to_csv(TAB_DIR / "checkpoint_status.csv", index=False)
    display(ck_df)
else:
    print(
        "[cảnh báo] không tìm thấy checkpoint GRKT nào — phiên sau sẽ phải train lại từ đầu."
    )

# đối chiếu fingerprint với cấu hình hiện tại (không raise: chỉ báo cáo)
mismatch = []
for r in ck_rows:
    if r["error"]:
        continue
    # Seed kỳ vọng đọc từ TÊN file, không phải luôn là SPLIT_SEED: vòng đa seed §8 ghi
    # GRKT-<đồ thị>-seed123.pt / -seed456.pt và fingerprint của chúng PHẢI khác 42. So với
    # SPLIT_SEED như bản trước thì hai checkpoint hợp lệ đó bị báo là "không khớp cấu hình".
    _m = re.search(r"-seed(\d+)\.pt$", r["file"])
    _want_seed = int(_m.group(1)) if _m else SPLIT_SEED
    for key, want in (
        ("seed", _want_seed),
        ("batch_size", BATCH_SIZE),
        ("d_hidden", D_HIDDEN),
    ):
        if r.get(key) is not None and int(r[key]) != int(want):
            mismatch.append(f"{r['dataset']}/{r['file']}: {key}={r[key]} != {want}")
    if (
        r["dataset"] == "junyi"
        and r.get("n_train")
        and int(r["n_train"]) > MAX_USERS * 1.2
    ):
        mismatch.append(
            f"junyi/{r['file']}: n_train={r['n_train']} > cap {MAX_USERS} (artifact cũ?)"
        )
print("\n[đối chiếu fingerprint]", "OK" if not mismatch else " | ".join(mismatch))
print(
    f"  * seed kỳ vọng lấy từ tên file: GRKT-<đồ thị>.pt là {SPLIT_SEED}, "
    "GRKT-<đồ thị>-seed<N>.pt là N (đa seed §8) — khác 42 là ĐÚNG, không phải artifact cũ."
)
if mismatch:
    print(
        ">> Artifact không khớp cấu hình hiện tại phải được cách ly/xoá thư mục output rồi chạy lại "
        "trước khi trích số (xem §0.3)."
    )

## 11. `report_summary.md` — sinh từ đĩa, không từ biến trong bộ nhớ

Mọi số trong mục này được đọc lại từ file đã ghi. Nhờ vậy báo cáo khớp với zip, và nếu một
mục thiếu thì báo cáo ghi "Thiếu" thay vì lấy số cũ trong RAM. Phần cuối là bảng đối chiếu
`YEU_CAU.md` §5–§8 và `GAP_LEVEL2.md` F1–F7 kèm **bằng chứng đọc từ đĩa**.


In [ ]:
REPORT_PATH = SAVE_DIR / "report_summary.md"


def _rj(path: Path):
    """Đọc JSON; trả None nếu thiếu / LFS pointer / hỏng."""
    if not path.exists():
        return None
    try:
        text = path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return None
    if (
        text.lstrip().startswith("version https://git-lfs")
        or "oid sha256" in text[:200]
    ):
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def _rc(path: Path):
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception:
        return None


def _md_table(df, cols=None, max_rows=14):
    if df is None or len(df) == 0:
        return "_(không có dữ liệu)_"
    d = df if cols is None else df[[c for c in cols if c in df.columns]]
    d = d.head(max_rows)
    lines = [
        "| " + " | ".join(str(c) for c in d.columns) + " |",
        "|" + "|".join("---" for _ in d.columns) + "|",
    ]
    for _, r in d.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |")
    return "\n".join(lines)


rep = []
rep.append("# Báo cáo Cấp độ 2 — GRKT (KDD 2024) và liên kết Cấp độ 1")
rep.append("")
rep.append(f"- Notebook: `{NOTEBOOK_VERSION}`")
rep.append(f"- Git commit: `{_head}`")
rep.append(f"- Sinh lúc (UTC): {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}")
rep.append(f"- Thiết bị: `{device}`")
rep.append("")

cfg = _rj(SAVE_DIR / "run_config.json") or {}
locked = cfg.get("locked") or {}
rep.append("## 0. Cấu hình đã khoá (run_config.json)")
rep.append("")
if locked:
    rep.append("```json")
    rep.append(json.dumps(locked, ensure_ascii=False, indent=2, default=str))
    rep.append("```")
else:
    rep.append("_(thiếu `run_config.json`)_")
rep.append("")

ds_dirs = sorted(
    d.name
    for d in SAVE_DIR.iterdir()
    if d.is_dir() and (d / "level2_metrics.csv").exists()
)
rep.append(f"## 1. Phạm vi và quy mô ({len(ds_dirs)} tập có bảng kết quả)")
rep.append("")
scale_rows = []
for ds in ds_dirs:
    summ = _rj(SAVE_DIR / ds / "summary.json") or {}
    stats = summ.get("data_stats") or {}
    split = _rj(SAVE_DIR / ds / "split.json") or {}
    n_tr = len(split.get("train_users", [])) if split else None
    n_va = len(split.get("val_users", [])) if split else None
    n_te = len(split.get("test_users", [])) if split else None
    graph = summ.get("graph_edges") or {}
    g_train = graph.get("train") or {}
    scale_rows.append(
        {
            "tập": ds,
            "người học": stats.get("users"),
            "tương tác": stats.get("interactions"),
            "khái niệm": stats.get("concepts", summ.get("concept_num")),
            "train/val/test": None if n_tr is None else f"{n_tr}/{n_va}/{n_te}",
            "cạnh rel/pre (train)": (
                None if not g_train else f"{g_train.get('rel')}/{g_train.get('pre')}"
            ),
        }
    )
rep.append(_md_table(pd.DataFrame(scale_rows)))
rep.append("")
if "junyi" in ds_dirs:
    summ_j = _rj(SAVE_DIR / "junyi" / "summary.json") or {}
    users_j = (summ_j.get("data_stats") or {}).get("users")
    rows_j = (summ_j.get("data_stats") or {}).get("interactions")
    if users_j and rows_j:
        rep.append(
            f"> **Junyi là TẬP CON**: {users_j:,} / {JUNYI_PUBLISHED['users']:,} người "
            f"({users_j / JUNYI_PUBLISHED['users']:.1%}), "
            f"{rows_j:,} / {JUNYI_PUBLISHED['rows']:,} lượt "
            f"({rows_j / JUNYI_PUBLISHED['rows']:.1%}). Không trích như 'toàn bộ Junyi'."
        )
        rep.append("")

rep.append("## 2. Kết quả chính (AUC / Accuracy / NLL)")
rep.append("")
best_rows = []
for ds in ds_dirs:
    m = _rc(SAVE_DIR / ds / "level2_metrics.csv")
    rep.append(f"### {ds}")
    rep.append("")
    rep.append(
        _md_table(
            m,
            cols=[
                "model",
                "graph",
                "auc",
                "acc",
                "loss",
                "val_auc",
                "n_test",
                "seed",
                "checkpoint_seed",
            ],
        )
    )
    rep.append("")
    if m is not None and len(m):
        b = m.sort_values("auc", ascending=False).iloc[0]
        best_rows.append((ds, str(b["model"]), str(b.get("graph")), float(b["auc"])))
        if "model_type" in m.columns:
            g = m[m["model_type"] == "GRKT"]
            base = m[m["model_type"] != "GRKT"]
            if len(g) and len(base):
                gb = g.sort_values("auc", ascending=False).iloc[0]
                bb = base.sort_values("auc", ascending=False).iloc[0]
                d_auc = float(gb["auc"]) - float(bb["auc"])
                rep.append(
                    f"- GRKT tốt nhất (`{gb['model']}`) − baseline tốt nhất (`{bb['model']}`) "
                    f"= **{d_auc:+.4f} AUC**."
                )
            if {"train", "zero"}.issubset(set(g["graph"].astype(str))):
                a_tr = float(g[g["graph"].astype(str) == "train"]["auc"].max())
                a_zr = float(g[g["graph"].astype(str) == "zero"]["auc"].max())
                rep.append(
                    f"- ΔAUC(train − zero) = **{a_tr - a_zr:+.4f}** "
                    f"({'có' if a_tr > a_zr else 'KHÔNG thấy'} lợi ích từ đồ thị)."
                )
        rep.append("")

rep.append("## 3. Quỹ đạo mastery m(u,c,t)")
rep.append("")
for ds in ds_dirs:
    t = _rc(SAVE_DIR / ds / "trajectory_metrics.csv")
    if t is None:
        rep.append(f"### {ds}\n\n_(thiếu trajectory_metrics.csv)_\n")
        continue
    rep.append(f"### {ds}")
    rep.append("")
    real = [
        "graph",
        "users",
        "precedence",
        "precedence_gap",
        "stability_spearman",
        "stability_changed_frac",
        "forget_slope",
        "pearson_dt_dm",
    ]
    inst = ["graph", "consistency_strengthen", "consistency_spread", "locality"]
    rep.append("**Tiêu chí thật (dùng làm bằng chứng):**")
    rep.append("")
    rep.append(_md_table(t, cols=real))
    rep.append("")
    rep.append("**Kiểm chứng cài đặt (kỳ vọng = 1,0; KHÔNG phải bằng chứng):**")
    rep.append("")
    rep.append(_md_table(t, cols=inst))
    rep.append("")
    if "precedence_gap" in t.columns and len(t):
        gap = float(t["precedence_gap"].mean())
        rep.append(
            f"- `precedence_gap` trung bình = **{gap:+.4f}** -> "
            + (
                "mastery khái niệm tiên quyết cao hơn khái niệm phụ thuộc (đúng chiều)."
                if gap > 0
                else "mastery khái niệm tiên quyết KHÔNG cao hơn — tiêu chí thứ tự chưa đạt; "
                "phải ghi đúng như vậy trong báo cáo, không suy diễn ngược."
            )
        )
    if "forget_slope" in t.columns and len(t):
        rep.append(
            f"- `forget_slope` trung bình = **{float(t['forget_slope'].mean()):.3e}** "
            "(âm = mastery giảm khi không luyện)."
        )
    rep.append("")

rep.append("## 4. Hiệu chỉnh xác suất")
rep.append("")
any_cal = False
for ds in ds_dirs:
    c = _rc(SAVE_DIR / ds / "calibration.csv")
    if c is None:
        continue
    any_cal = True
    rep.append(f"### {ds}")
    rep.append("")
    rep.append(_md_table(c))
    rep.append("")
    if "ece" in c.columns and len(c):
        b = c.sort_values("ece").iloc[0]
        rep.append(f"- ECE thấp nhất: `{b.get('model')}` = **{float(b['ece']):.4f}**.")
        rep.append("")
if not any_cal:
    rep.append("_(thiếu calibration.csv)_\n")

rep.append("## 5. Đa seed")
rep.append("")
any_ms = False
for ds in ds_dirs:
    ms = _rc(SAVE_DIR / ds / "multiseed_metrics.csv")
    if ms is None:
        continue
    any_ms = True
    rep.append(f"### {ds}")
    rep.append("")
    rep.append(_md_table(ms))
    rep.append("")
if not any_ms:
    rep.append(
        "_(không có multiseed_metrics.csv — Junyi chỉ chạy 1 seed và điều đó được ghi rõ)_\n"
    )

rep.append("## 6. Liên kết Cấp độ 1")
rep.append("")
link = _rj(SAVE_DIR / "diagnostics" / "level1_linkage.json")
lk = (link or {}).get("legacy_ckpt") or {}
if link is None:
    rep.append("_(thiếu diagnostics/level1_linkage.json)_")
else:
    rep.append(
        f"- `split.json` C1: **{link.get('level1_split_status')}** | "
        f"C2: **{link.get('level2_split_status')}**"
    )
    rep.append(
        f"- Tập người học trùng Cấp độ 1: **{link.get('same_split')}** "
        f"(max_seq_len={link.get('max_seq_len')}, split_seed={link.get('split_seed')})"
    )
    if link.get("split_rows"):
        rep.append("")
        rep.append(_md_table(pd.DataFrame(link["split_rows"])))
    rep.append("")
    bc = pd.DataFrame(link.get("baseline_comparison") or [])
    if len(bc):
        rep.append("**Đường nền train lại seed 42 so với `level1_metrics.csv`:**")
        rep.append("")
        rep.append(_md_table(bc))
        rep.append("")
        rep.append(
            f"> same_weights={link.get('same_weights')} -> dung sai "
            f"{0.005 if link.get('same_weights') else 0.02}. Δ nhỏ là kiểm tra split/cửa sổ, "
            "không chứng minh trọng số trùng Cấp độ 1."
        )
        rep.append("")
    rep.append(f"**Checkpoint Cấp độ 1 đã commit (GAP F2):** {lk.get('verdict')}")
    if lk.get("rows"):
        rep.append("")
        rep.append(_md_table(pd.DataFrame(lk["rows"])))
    rep.append("")

rep.append("## 7. Dashboard người học")
rep.append("")
dash_path = SAVE_DIR / "dashboard" / "learner_dashboard_summary.csv"
dsum = _rc(dash_path)
if dsum is None:
    rep.append("_(thiếu dashboard/learner_dashboard_summary.csv)_")
else:
    rep.append(f"- {len(dsum)} người học, cột: {', '.join(dsum.columns)}")
    rep.append("")
    rep.append(_md_table(dsum, max_rows=10))
rep.append("")

rep.append("## 8. Đối chiếu yêu cầu và khoảng trống")
rep.append("")
check = []
metrics_all = {ds: _rc(SAVE_DIR / ds / "level2_metrics.csv") for ds in ds_dirs}
traj_all = {ds: _rc(SAVE_DIR / ds / "trajectory_metrics.csv") for ds in ds_dirs}
ms_assist = _rc(SAVE_DIR / "assist2009" / "multiseed_metrics.csv")
m_assist = metrics_all.get("assist2009")
has_zero = bool(
    m_assist is not None
    and "graph" in m_assist.columns
    and (m_assist["graph"].astype(str) == "zero").any()
)
has_cal = any(_rc(SAVE_DIR / ds / "calibration.csv") is not None for ds in ds_dirs)
has_dash = dash_path.exists()
has_cfg = (SAVE_DIR / "run_config.json").exists() and (
    SAVE_DIR / "run_sessions.json"
).exists()
has_env = (SAVE_DIR / "environment.txt").exists()
has_curves = any(FIG_DIR.glob("learning_curves_*.png"))
real_ok = (
    all(
        t is not None and all(c in t.columns for c in REAL_CRITERIA)
        for t in traj_all.values()
    )
    if traj_all
    else False
)
n_seed_min = (
    int(ms_assist["n_seed"].min())
    if (ms_assist is not None and "n_seed" in ms_assist.columns and len(ms_assist))
    else 0
)
same_split = bool(link.get("same_split")) if link else False
prov_disk = _rj(SAVE_DIR / "run_provenance.json") or {}
junyi_users = (
    (_rj(SAVE_DIR / "junyi" / "summary.json") or {}).get("data_stats") or {}
).get("users")
epochs_used = sorted(
    {
        int(r.get("epochs_run") or 0)
        for ds in prov_disk.values()
        for r in (ds.get("models") or [])
        if r.get("epochs_run")
    }
)

check.append(
    (
        "§5 GRKT trên ≥2 tập công khai",
        f"{len(ds_dirs)} tập: {', '.join(ds_dirs)}",
        "Đạt" if len(ds_dirs) >= 2 else "Thiếu",
    )
)
check.append(
    (
        "§5 dùng split theo người học của C1",
        f"same_split={same_split}, seed={locked.get('split_seed')}, max_seq_len={locked.get('max_seq_len')}",
        "Đạt" if same_split else "Một phần (cùng seed/hàm, thiếu bằng chứng file)",
    )
)
check.append(
    (
        "§6.2 đồ thị chỉ từ train",
        "graphs_train_only.npz + leak_report.txt",
        (
            "Đạt"
            if all((SAVE_DIR / ds / "graphs_train_only.npz").exists() for ds in ds_dirs)
            else "Thiếu"
        ),
    )
)
check.append(
    (
        "§6.4 train/val/test rời nhau",
        f"{len(link.get('split_rows') or []) if link else 0} dòng đối chiếu",
        "Đạt" if same_split else "Cần kiểm tra split.json",
    )
)
check.append(
    (
        "§7 AUC / ACC / NLL",
        "level2_metrics.csv có đủ auc/acc/loss",
        (
            "Đạt"
            if all(
                m is not None and {"auc", "acc", "loss"}.issubset(m.columns)
                for m in metrics_all.values()
            )
            else "Thiếu"
        ),
    )
)
check.append(
    (
        "§7 mean ± std ≥3 seed",
        f"assist2009 n_seed min={n_seed_min}",
        "Đạt" if n_seed_min >= 3 else "Thiếu",
    )
)
check.append(
    (
        "§7 ablation đồ thị (train vs zero)",
        f"assist2009 có zero={has_zero}",
        "Đạt" if has_zero else "Thiếu",
    )
)
check.append(
    (
        "§7 một phân tích nâng cao",
        f"quỹ đạo thật={'có' if real_ok else 'thiếu'}, calibration={'có' if has_cal else 'thiếu'}",
        "Đạt" if (real_ok and has_cal) else "Một phần",
    )
)
check.append(
    (
        "§8 dashboard người học C2",
        f"learner_dashboard_summary.csv={'có' if has_dash else 'thiếu'}",
        "Đạt" if has_dash else "Thiếu",
    )
)
check.append(
    (
        "GAP F1 tiêu chí quỹ đạo không đóng băng",
        "tách bảng THẬT vs KIỂM CHỨNG CÀI ĐẶT",
        "Đạt" if real_ok else "Thiếu",
    )
)
check.append(
    (
        "GAP F2 seed checkpoint C1",
        (lk.get("verdict") or "chưa đo"),
        (
            "Đạt (đã đo; bảng chính dùng đường nền train lại)"
            if lk.get("evaluated")
            else "Một phần"
        ),
    )
)
check.append(
    (
        "GAP F3 artifact Junyi khớp mô tả",
        f"junyi users={junyi_users}, cap={MAX_USERS}",
        (
            "Không áp dụng (không chạy Junyi)"
            if "junyi" not in ds_dirs
            else (
                "Đạt (tập con, cap khớp)"
                if junyi_users and int(junyi_users) <= MAX_USERS * 1.05
                else "KHÔNG khớp cap — kiểm tra artifact cũ"
            )
        ),
    )
)
check.append(
    (
        "GAP F4 GRKT cùng ngân sách epoch với baseline",
        f"epochs_run={epochs_used or 'n/a'}, cap={locked.get('epochs')}, patience={locked.get('patience')}",
        (
            "Đạt"
            if (epochs_used and max(epochs_used) > 2)
            else "Cần kiểm tra run_provenance.json"
        ),
    )
)
check.append(
    (
        "GAP F5 config/history/provenance",
        f"run_config={'có' if (SAVE_DIR/'run_config.json').exists() else 'thiếu'}, "
        f"run_sessions={'có' if (SAVE_DIR/'run_sessions.json').exists() else 'thiếu'}, "
        f"environment={'có' if has_env else 'thiếu'}, "
        f"learning_curves={'có' if has_curves else 'thiếu'}",
        "Đạt" if (has_cfg and has_env) else "Thiếu",
    )
)
check.append(
    (
        "GAP F6 biến thể đồ thị",
        f"assist2009 variants={locked.get('variants_by_dataset', {}).get('assist2009')}",
        "Đạt" if has_zero else "Thiếu",
    )
)
check.append(
    (
        "GAP F7 dashboard C2",
        f"{'có' if has_dash else 'thiếu'}",
        "Đạt" if has_dash else "Thiếu",
    )
)
rep.append(
    _md_table(
        pd.DataFrame(check, columns=["Mục", "Bằng chứng (đọc từ đĩa)", "Kết luận"]),
        max_rows=30,
    )
)
rep.append("")

rep.append("## 9. Kết luận tự động")
rep.append("")
if best_rows:
    for ds, model, graph, auc in best_rows:
        rep.append(
            f"- **{ds}**: mô hình tốt nhất theo Test AUC là `{model}`"
            + (f" (đồ thị `{graph}`)" if graph and graph != "nan" else "")
            + f", AUC = **{auc:.4f}**."
        )
else:
    rep.append("- Không có bảng kết quả để kết luận.")
if "assist2009" in metrics_all and m_assist is not None and has_zero:
    _gt = m_assist[m_assist["graph"].astype(str) == "train"]["auc"]
    _gz = m_assist[m_assist["graph"].astype(str) == "zero"]["auc"]
    if len(_gt) and len(_gz):
        a_tr, a_zr = float(_gt.max()), float(_gz.max())
        rep.append(
            f"- **Vai trò đồ thị trên assist2009**: ΔAUC = **{a_tr - a_zr:+.4f}**; "
            + (
                "có lợi ích đo được."
                if a_tr > a_zr
                else "không thấy lợi ích — phải ghi đúng, kèm lưu ý một seed."
            )
        )
if has_dash:
    rep.append(
        "- **Dashboard** đã có đủ xác suất từng bước, nhãn, mastery trước/sau lan truyền; "
        "Junyi chỉ là tập con nên không dùng để suy rộng."
    )
rep.append("")

rep.append("## 10. Hạn chế phải nêu trong báo cáo")
rep.append("")
rep.append(
    "1. Junyi là **tập con** (người học lấy mẫu có seed) — mọi số Junyi chỉ có giá trị trên tập con."
)
rep.append("2. Chọn 200 khái niệm Junyi dùng thống kê **toàn bộ log** (transductive).")
rep.append(
    "3. `consistency_strengthen`, `consistency_spread`, `locality` ≡ 1 do cấu trúc mô hình, "
    "không phải bằng chứng đồ thị (GAP F1)."
)
rep.append(
    "4. Ablation đồ thị và đa seed chỉ chạy trên assist2009; Junyi 1 seed, 1 đồ thị."
)
rep.append(
    "5. ΔAUC(train − zero) chỉ từ một seed: chưa đủ để kết luận nhân quả về đồ thị."
)
rep.append(
    "6. Checkpoint Cấp độ 1 đã commit không chứng minh được seed 42; bảng chính dùng đường nền "
    "train lại, checkpoint cũ chỉ để chẩn đoán (§5.2)."
)
rep.append("")

REPORT_PATH.write_text("\n".join(rep), encoding="utf-8")
print(f"Đã ghi {REPORT_PATH} ({len(rep)} dòng).\n")
display(pd.DataFrame(check, columns=["Mục", "Bằng chứng", "Kết luận"]))
print("\n--- trích đầu báo cáo ---")
print("\n".join(rep[: min(len(rep), 42)]))

## 12. Đóng gói kép để tải về

Kaggle xoá `/kaggle/working` khi phiên kết thúc, nên phải đóng gói **trước khi tắt**. Ngoài ra §3
chạy `PeriodicPackager` ở luồng nền: cứ `PACK_EVERY_MIN` phút lại ghi `level2_moderate_output.zip`
(nguyên tử: `<tên>.part` rồi `os.replace`), nên kể cả khi phiên bị kill cứng — hết quota GPU, rớt
kết nối, quá 12 h — tổn thất tối đa là `PACK_EVERY_MIN` phút và một lần đóng gói dang dở không
phá được zip tốt của lần trước.

Hai zip, mục đích khác nhau:

| Zip | Nội dung | Dùng để |
|---|---|---|
| `level2_moderate_output.zip` | mọi thứ: `GRKT-*.pt` (mọi biến thể/seed), `baselines/*.pt`, `ckpt_*.pt`, và `*_canonical.csv` + `.meta.json` nếu ≤ `max_mb`; luồng nền tự cập nhật mỗi `PACK_EVERY_MIN` phút, cộng thêm sau mỗi tập / khi dừng / khi lỗi ở §3 | resume phiên sau; `EVAL_ONLY=True` để đánh giá lại mà không train |
| `level2_moderate_report_pack.zip` | số liệu, bảng, hình, JSON, dashboard — **không** `.pt`/`.npz`, **không** CSV chuẩn hoá (`LIGHT_SKIP`) | gửi kèm báo cáo, nhẹ và đọc được ngay |

CSV chuẩn hoá nằm trong zip chính là thứ làm cho gói **tự chứa**: có nó thì `EVAL_ONLY` không phải
attach lại Junyi thô ~3 GB rồi chuẩn hoá hàng chục phút, và `_quarantine_junyi` ở §0.3 đọc
`.meta.json` để xác nhận CSV đó đúng cap `MAX_USERS`. Bản chạy **toàn bộ** Junyi (~1 GB) vẫn bị
`max_mb=150` loại; khi đó dòng `bỏ N file` in ra lý do kèm hậu quả, và §2 tái sinh từ file gốc.
`README_pack.txt` ghi rõ phiên bản notebook, commit, quy mô từng tập và cảnh báo tập con Junyi.


In [ ]:
from level2.archive import LIGHT_SKIP, package_output

readme_lines = [
    "GÓI KẾT QUẢ CẤP ĐỘ 2 (GRKT) — Knowledge-Tracing",
    "=" * 60,
    f"notebook     : {NOTEBOOK_VERSION}",
    f"git commit   : {_head}",
    f"xuất lúc     : {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}",
    f"thiết bị     : {device}",
    f"thư mục gốc  : {SAVE_DIR.name}/",
    "",
    "PHẠM VI SỐ LIỆU (đọc trước khi trích dẫn)",
    f"  assist2009 : toàn bộ split Cấp độ 1, split_seed={SPLIT_SEED}, max_seq_len={MAX_SEQ_LEN}",
    f"  junyi      : TẬP CON tối đa {MAX_USERS} người học (KHÔNG phải toàn bộ 72.630 người).",
    "               Lọc >=10 tương tác, giữ 200 bài phổ biến nhất, lấy mẫu người học có seed.",
    "               Chọn 200 khái niệm dùng thống kê trên toàn bộ log (transductive).",
    "",
    "CẤU TRÚC",
    "  <tập>/level2_metrics.csv        bảng DKT / GKT / GRKT (seed + checkpoint_seed)",
    "  <tập>/summary.json              quy mô dữ liệu, số cạnh đồ thị, cấu hình",
    "  <tập>/split.json                danh sách người học train/val/test",
    "  <tập>/trajectory_metrics.csv    tiêu chí quỹ đạo THẬT và kiểm chứng cài đặt",
    "  <tập>/calibration.csv           ECE / Brier",
    "  <tập>/multiseed_metrics.csv     mean ± std trên >=3 seed (nếu bật)",
    "  <tập>/GRKT-<graph>.pt            trọng số GRKT của từng biến thể/seed",
    "  <tập>/baselines/<tên>.pt         trọng số DKT/GKT train lại cho EVAL_ONLY",
    "  <tập>/trajectories/             npz + mastery csv + heatmap",
    "  dashboard/                      dashboard người học (JSON/CSV)",
    "  diagnostics/level1_linkage.json đối chiếu split + AUC với Cấp độ 1",
    "  figures/, tables/               hình và bảng cho báo cáo",
    "  report_summary.md               báo cáo sinh tự động từ đĩa",
    "  run_config.json                 cấu hình đã khoá",
    "  run_sessions.json               lịch sử phiên, history từng epoch, resume",
    "  environment.txt                 phiên bản thư viện / GPU",
    "",
    "ĐÁNH GIÁ LẠI KHÔNG TRAIN:",
    "  Đặt EVAL_ONLY=True, attach level2_moderate_output.zip, Run All; §3 nạp GRKT +",
    "  baseline từ zip và chỉ đánh giá. Thiếu checkpoint nào notebook báo đúng đường dẫn.",
    "  Zip tự chứa cả <tập>/*_canonical.csv + .meta.json nên KHÔNG cần attach dữ liệu gốc,",
    "  trừ khi CSV đó bị loại vì lớn hơn max_mb (xem dòng 'bỏ N file' khi đóng gói).",
    "",
    "GHI CHÚ",
    "  - consistency_strengthen / consistency_spread / locality = 1,0 là KIỂM CHỨNG CÀI ĐẶT,",
    "    không phải bằng chứng vai trò đồ thị (GAP F1).",
    "  - Zip report_pack KHÔNG có .pt/.npz và KHÔNG có CSV chuẩn hoá -> không resume được,",
    "    chỉ để đọc số liệu.",
    "  - Zip output được luồng nền cập nhật mỗi PACK_EVERY_MIN phút và ghi nguyên tử, nên",
    "    phiên bị kill cứng vẫn để lại zip không cũ hơn một chu kỳ đóng gói.",
]
(SAVE_DIR / "README_pack.txt").write_text("\n".join(readme_lines), encoding="utf-8")
print("\n".join(readme_lines))

out_zip = package_output(SAVE_DIR, ZIP_PATH, max_mb=150, keep_ckpts=True)
report_zip = package_output(
    SAVE_DIR, REPORT_ZIP_PATH, max_mb=60, keep_ckpts=False, skip_names=LIGHT_SKIP
)
print(f"\n>> TẢI VỀ NGAY trước khi tắt phiên:\n   {out_zip}\n   {report_zip}")
if IS_KAGGLE:
    print(
        "   (Kaggle: tab Output -> tải file .zip; hoặc Save Version rồi Download Output.)"
    )

## 13. Sự cố thường gặp và cách xử lý

| Hiện tượng | Nguyên nhân | Xử lý |
|---|---|---|
| `FileNotFoundError` Junyi ở §2/§3 | Chưa Agree/Subscribe dataset | Kaggle → dataset Junyi → Agree → Add Input; hoặc đặt `DATASETS=["assist2009"]` |
| `RuntimeError: run_config.json khác cấu hình` | Resume với `EPOCHS`/`max_users`/`split_seed` khác | Đổi `SAVE_DIR`/`TOP`, hoặc xoá thư mục output, hoặc (không khuyến nghị) `ALLOW_CONFIG_MISMATCH=True` |
| `RuntimeError: có checkpoint ... không có run_config.json` | Attach zip từ notebook khác | Chỉ attach `level2_moderate_output.zip` của notebook này |
| `TrainingPaused` giữa §3 | Hết `TIME_BUDGET_MIN` | Bình thường: checkpoint đã ghi + zip đã đóng. Phiên sau attach zip, giữ `RESUME_GRKT_CKPT=True` |
| CUDA OOM khi train GRKT | `batch_size × K × max_seq_len` quá lớn | Giảm `BATCH_SIZE` trước, rồi `top_concepts`; không cần giảm số người |
| split.json là file vài trăm byte | Git LFS chưa fetch (clone `--depth 1`) | `git lfs pull --include outputs/level1/*`; notebook đã thử tự động, nếu vẫn lỗi thì bỏ đối chiếu split và ghi rõ |
| Junyi bị chuyển vào `junyi/stale_*` | Attach nhầm artifact chạy toàn bộ | Đúng như thiết kế: không xoá, chỉ cách ly. Xoá thủ công nếu chắc chắn không cần |
| Zip quá lớn / thiếu file | `max_mb` hoặc `keep_ckpts` | Zip output giữ checkpoint; report pack bỏ `.pt`/`.npz`. Tăng `max_mb` nếu cần mang hình lớn |
| Kết quả khác lần trước dù cùng cấu hình | Thiếu seed / resume khác nhánh | Xem `seed`, `checkpoint_seed`, `run_sessions.json`; không so hai bảng khác cấu hình |
| Muốn **chỉ đánh giá + dashboard** từ output cũ | Đang để `EVAL_ONLY=False` | Đặt `EVAL_ONLY=True`, attach `level2_moderate_output.zip`, Run All; thiếu checkpoint nào notebook báo đúng đường dẫn |


## 14. Việc cần làm sau khi chạy xong

1. Tải **cả hai** zip ở §12; `level2_moderate_output.zip` là bản có thể resume, report pack để đọc.
2. Mở `report_summary.md`: bảng §8 phải toàn "Đạt"; mục nào "Một phần"/"Thiếu" thì chạy bù trước
   khi trích số.
3. Trong báo cáo, luôn ghi Junyi là **tập con** và trích đúng dòng "Junyi là TẬP CON" ở §1.
4. Không dùng `consistency_*`/`locality` làm bằng chứng đồ thị; dùng `precedence_gap`,
   `stability_spearman`, `forget_slope`, `pearson_dt_dm` và ΔAUC(train − zero).
5. Nếu cần số Junyi toàn bộ: đặt `JUNYI_MAX_USERS=None` và chạy nhiều phiên với resume — nhưng
   đó là một lần chạy khác, phải đổi `TOP`/`SAVE_DIR` để không trộn cấu hình.
